# EnMAP SS-MSAE V3 — Final Research Pipeline

### Kaggle settings
1. Add the original EnMAP `.rar` file as an Input Dataset.
2. Set **Accelerator = GPU T4**.
3. Set **Internet = ON** so the public hyperspectral benchmark datasets can be downloaded automatically.
4. Run **Run All**.
5. At the end, download and send back:
   `ENMAP_SSMAE_V3_FINAL_RESULTS_BUNDLE.zip`

### What this V3 adds
- Full EnMAP paper-ready analysis from V2.
- 13 ABU anomaly-detection benchmark scenes.
- HYDICE Urban benchmark.
- San Diego AVIRIS benchmark when its public files are reachable.
- RX, PCA-RX, AE, MSAE, derivative-aware MSAE, and the proposed spectral-spatial fusion detector.
- ROC-AUC, PR-AUC, F1@P99, Precision/Recall@P99, FPR@95% TPR, post-hoc oracle F1.
- Ablation, multi-seed robustness, Wilcoxon paired tests, bootstrap 95% confidence intervals, runtime/parameter analysis.
- Publication figures in PNG + PDF, per-scene maps, curve CSVs, and a multi-sheet Excel workbook.

**Scientific safeguard:** benchmark ground truth is used only after anomaly-score generation for evaluation; it is not used for unsupervised training or fusion-weight tuning.


In [ ]:
# ENMAP AAMSAE PAPER PIPELINE
# Kaggle-ready, label-free hyperspectral AI workflow for EnMAP L2A geology/mineral anomaly discovery.

import os, re, json, math, time, random, shutil, subprocess, ctypes, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
from sklearn.covariance import LedoitWolf

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset



In [ ]:
# ============================================================
# 0) CONFIGURATION
# ============================================================
SEED = 42
DATA_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('/mnt/data')
WORK_ROOT = Path('/kaggle/working/enmap_aamsae_paper') if Path('/kaggle/working').exists() else Path('/mnt/data/enmap_aamsae_paper')
EXTRACT_DIR = WORK_ROOT / 'extracted'
OUT_DIR = WORK_ROOT / 'results'
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Paper-quality defaults. Reduce only for a quick debug run.
TRAIN_SAMPLES = int(os.environ.get('ENMAP_TRAIN_SAMPLES', '60000'))
EVAL_SAMPLES = int(os.environ.get('ENMAP_EVAL_SAMPLES', '12000'))
FULL_EPOCHS = int(os.environ.get('ENMAP_FULL_EPOCHS', '25'))
ABLATION_EPOCHS = int(os.environ.get('ENMAP_ABLATION_EPOCHS', '12'))
BATCH_SIZE = int(os.environ.get('ENMAP_BATCH_SIZE', '512'))
MAP_BATCH = int(os.environ.get('ENMAP_MAP_BATCH', '4096'))
LATENT_DIM = 32
MASK_RATIO = 0.35
K_RANGE = list(range(4, 11))
TOP_ANOMALIES = 100
RUN_ABLATION = os.environ.get('ENMAP_RUN_ABLATION', '1') == '1'
MAP_STRIDE = int(os.environ.get('ENMAP_MAP_STRIDE', '1'))  # keep 1 for paper figures

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cpu' and MAP_STRIDE == 1:
    print('WARNING: CPU detected. Full-resolution inference can be slow. Set ENMAP_MAP_STRIDE=2 for a quick run.')

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('DEVICE:', DEVICE)
print('DATA_ROOT:', DATA_ROOT)
print('WORK_ROOT:', WORK_ROOT)



In [ ]:
# ============================================================
# 1) RAR EXTRACTION (works with system tools OR libarchive)
# ============================================================
def extract_with_libarchive(rar_path: Path, dest: Path):
    lib = None
    for name in ('libarchive.so', 'libarchive.so.13'):
        try:
            lib = ctypes.CDLL(name); break
        except OSError:
            pass
    if lib is None:
        raise RuntimeError('libarchive not available')

    lib.archive_read_new.restype = ctypes.c_void_p
    lib.archive_read_support_filter_all.argtypes = [ctypes.c_void_p]
    lib.archive_read_support_format_all.argtypes = [ctypes.c_void_p]
    lib.archive_read_open_filename.argtypes = [ctypes.c_void_p, ctypes.c_char_p, ctypes.c_size_t]
    lib.archive_read_next_header.argtypes = [ctypes.c_void_p, ctypes.POINTER(ctypes.c_void_p)]
    lib.archive_entry_pathname.argtypes = [ctypes.c_void_p]
    lib.archive_entry_pathname.restype = ctypes.c_char_p
    lib.archive_read_data.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_size_t]
    lib.archive_read_data.restype = ctypes.c_ssize_t
    lib.archive_read_data_skip.argtypes = [ctypes.c_void_p]
    lib.archive_read_free.argtypes = [ctypes.c_void_p]

    a = lib.archive_read_new()
    lib.archive_read_support_filter_all(a)
    lib.archive_read_support_format_all(a)
    ret = lib.archive_read_open_filename(a, str(rar_path).encode(), 1024 * 1024)
    if ret != 0:
        lib.archive_read_free(a)
        raise RuntimeError(f'libarchive could not open {rar_path}')

    entry = ctypes.c_void_p()
    while lib.archive_read_next_header(a, ctypes.byref(entry)) == 0:
        rel = lib.archive_entry_pathname(entry).decode('utf-8', 'replace')
        # Prevent path traversal
        rel = rel.replace('..', '_').lstrip('/\\')
        target = dest / rel
        if rel.endswith('/') or Path(rel).suffix == '':
            target.mkdir(parents=True, exist_ok=True)
            lib.archive_read_data_skip(a)
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        with open(target, 'wb') as f:
            buf = ctypes.create_string_buffer(4 * 1024 * 1024)
            while True:
                n = lib.archive_read_data(a, buf, len(buf))
                if n <= 0: break
                f.write(buf.raw[:n])
    lib.archive_read_free(a)


def ensure_extracted(root: Path, dest: Path):
    hdrs = list(root.rglob('*SPECTRAL_IMAGE.HDR')) + list(root.rglob('*spectral_image.hdr'))
    if hdrs:
        return root
    rar_files = list(root.rglob('*.rar')) + list(root.rglob('*.RAR'))
    if not rar_files:
        raise FileNotFoundError('No EnMAP SPECTRAL_IMAGE.HDR and no .rar file found under DATA_ROOT.')
    rar_path = max(rar_files, key=lambda p: p.stat().st_size)
    print('RAR found:', rar_path)
    print('Extracting...')
    # Try command-line tools first
    commands = []
    if shutil.which('unrar'):
        commands.append(['unrar', 'x', '-o+', str(rar_path), str(dest) + '/'])
    if shutil.which('7z'):
        commands.append(['7z', 'x', '-y', f'-o{dest}', str(rar_path)])
    if shutil.which('unar'):
        commands.append(['unar', '-f', '-o', str(dest), str(rar_path)])
    for cmd in commands:
        try:
            subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            if list(dest.rglob('*SPECTRAL_IMAGE.HDR')):
                print('Extraction completed with:', cmd[0])
                return dest
        except Exception:
            pass
    print('System extractor unavailable/failed; using libarchive fallback...')
    extract_with_libarchive(rar_path, dest)
    return dest

DATA_DIR = ensure_extracted(DATA_ROOT, EXTRACT_DIR)



In [ ]:
# ============================================================
# 2) FIND ENMAP FILES + PARSE ENVI HEADER
# ============================================================
hdr_candidates = list(DATA_DIR.rglob('*SPECTRAL_IMAGE.HDR'))
if not hdr_candidates:
    raise FileNotFoundError('SPECTRAL_IMAGE.HDR not found after extraction.')
HDR = hdr_candidates[0]
BSQ = HDR.with_suffix('.BSQ')
if not BSQ.exists():
    matches = list(HDR.parent.glob('*SPECTRAL_IMAGE.BSQ'))
    if not matches: raise FileNotFoundError('SPECTRAL_IMAGE.BSQ not found next to HDR.')
    BSQ = matches[0]
META = next(iter(HDR.parent.glob('*METADATA.XML')), None)

print('HDR :', HDR)
print('BSQ :', BSQ)
print('META:', META)

hdr_text = HDR.read_text(errors='ignore')

def scalar(key, cast=str, default=None):
    m = re.search(rf'(?im)^\s*{re.escape(key)}\s*=\s*([^\r\n]+)', hdr_text)
    if not m: return default
    v = m.group(1).strip()
    try: return cast(v)
    except Exception: return default

def brace_list(key, cast=float):
    m = re.search(rf'(?is)\b{re.escape(key)}\s*=\s*\{{(.*?)\}}', hdr_text)
    if not m: return None
    vals = [x.strip() for x in m.group(1).replace('\n',' ').split(',') if x.strip()]
    return [cast(x) for x in vals]

samples = scalar('samples', int)
lines = scalar('lines', int)
bands = scalar('bands', int)
interleave = scalar('interleave', str, 'bsq').lower()
data_type = scalar('data type', int, 2)
byte_order = scalar('byte order', int, 0)
nodata = scalar('data ignore value', float, -32768)
wavelengths = np.array(brace_list('wavelength', float), dtype=np.float32)
gains = brace_list('data gain values', float)
offsets = brace_list('data offset values', float)
map_info = brace_list('map info', str)

if wavelengths is None or len(wavelengths) != bands:
    raise ValueError('Could not parse wavelengths from HDR.')
if gains is None or len(gains) != bands: gains = [1.0] * bands
if offsets is None or len(offsets) != bands: offsets = [0.0] * bands
gains = np.asarray(gains, dtype=np.float32)
offsets = np.asarray(offsets, dtype=np.float32)

ENVI_DTYPES = {
    1:'u1', 2:'i2', 3:'i4', 4:'f4', 5:'f8', 12:'u2', 13:'u4', 14:'i8', 15:'u8'
}
if data_type not in ENVI_DTYPES:
    raise ValueError(f'Unsupported ENVI data type: {data_type}')
endian = '<' if byte_order == 0 else '>'
dtype = np.dtype(endian + ENVI_DTYPES[data_type])
if interleave != 'bsq':
    raise NotImplementedError(f'This pipeline expects BSQ interleave, found: {interleave}')

cube = np.memmap(BSQ, dtype=dtype, mode='r', shape=(bands, lines, samples))
print('Cube:', cube.shape, cube.dtype)
print(f'Wavelength: {wavelengths.min():.1f}–{wavelengths.max():.1f} nm')

# Parse map info if available
x0=y0=dx=dy=zone=None; hemisphere=None
try:
    if map_info and len(map_info) >= 10:
        x0 = float(map_info[3]); y0 = float(map_info[4]); dx = float(map_info[5]); dy = float(map_info[6])
        zone = int(float(map_info[7])); hemisphere = str(map_info[8]).strip()
except Exception:
    pass



In [ ]:
# ============================================================
# 3) BAND / FOOTPRINT QC + TRAIN/EVAL SAMPLING
# ============================================================
qc_stride = max(8, int(round(max(lines, samples) / 140)))
coarse = np.asarray(cube[:, ::qc_stride, ::qc_stride])
nodata_frac = np.mean(coarse == nodata, axis=(1,2))
# Automatically remove fully/near-fully missing bands.
good_band_mask = nodata_frac < 0.98
good_bands = np.where(good_band_mask)[0]
wl = wavelengths[good_bands]
gain_good = gains[good_bands]
offset_good = offsets[good_bands]
print('Usable bands:', len(good_bands), '/', bands)
print('Removed bands:', (np.where(~good_band_mask)[0] + 1).tolist())

# Footprint mask from several distributed usable bands.
probe_idx = np.unique(np.linspace(0, len(good_bands)-1, 7).round().astype(int))
valid_mask = np.ones((lines, samples), dtype=bool)
for j in probe_idx:
    valid_mask &= (cube[good_bands[j]] != nodata)
valid_coords = np.argwhere(valid_mask)
print('Valid pixels:', len(valid_coords), '/', lines*samples)
if len(valid_coords) < 1000:
    raise RuntimeError('Too few valid pixels detected.')

rng = np.random.default_rng(SEED)
n_total = min(len(valid_coords), TRAIN_SAMPLES + EVAL_SAMPLES)
sel = rng.choice(len(valid_coords), size=n_total, replace=False)
train_n = min(TRAIN_SAMPLES, max(1000, int(n_total*0.8)))
train_coords = valid_coords[sel[:train_n]]
eval_coords = valid_coords[sel[train_n:]]
if len(eval_coords) > EVAL_SAMPLES: eval_coords = eval_coords[:EVAL_SAMPLES]
if len(eval_coords) < 1000:
    # split train if scene is small
    eval_coords = train_coords[-1000:]
    train_coords = train_coords[:-1000]
print('Train pixels:', len(train_coords), ' Eval pixels:', len(eval_coords))


def read_spectra(coords):
    rows, cols = coords[:,0], coords[:,1]
    arr = np.empty((len(coords), len(good_bands)), dtype=np.float32)
    for j, b in enumerate(good_bands):
        arr[:, j] = np.asarray(cube[b, rows, cols], dtype=np.float32) * gain_good[j] + offset_good[j]
    return arr

X_train_raw = read_spectra(train_coords)
X_eval_raw = read_spectra(eval_coords)

# Robust clip / standardization fitted only on training sample.
q_lo = np.nanpercentile(X_train_raw, 0.5, axis=0)
q_hi = np.nanpercentile(X_train_raw, 99.5, axis=0)
X_train_clip = np.clip(X_train_raw, q_lo, q_hi)
mu = X_train_clip.mean(axis=0)
sigma = X_train_clip.std(axis=0) + 1e-6

def normalize_spectra(X):
    return ((np.clip(X, q_lo, q_hi) - mu) / sigma).astype(np.float32)

X_train = normalize_spectra(X_train_raw)
X_eval = normalize_spectra(X_eval_raw)

# Geology-oriented spectral weighting: VNIR iron-related region and SWIR mineral absorption region.
# This is a soft weighting, not a mineral label.
band_weight = np.ones(len(wl), dtype=np.float32)
band_weight[(wl >= 700) & (wl <= 1000)] = 1.35
band_weight[(wl >= 2000) & (wl <= 2400)] = 1.75
# De-emphasize edges where signal can be less stable.
band_weight[(wl < 430) | (wl > 2420)] *= 0.8
# Adjacent-derivative loss should not bridge large wavelength gaps.
delta_wl = np.diff(wl)
deriv_valid = (delta_wl < 25).astype(np.float32)
wl_norm = ((wl - wl.min()) / (wl.max() - wl.min()) * 2 - 1).astype(np.float32)

# Save data summary early.
summary = {
    'samples': int(samples), 'lines': int(lines), 'bands_total': int(bands), 'bands_usable': int(len(good_bands)),
    'wavelength_min_nm': float(wl.min()), 'wavelength_max_nm': float(wl.max()),
    'valid_pixels': int(len(valid_coords)), 'train_pixels': int(len(train_coords)), 'eval_pixels': int(len(eval_coords)),
    'nodata_value': float(nodata), 'device': str(DEVICE), 'map_stride': int(MAP_STRIDE),
    'x0': x0, 'y0': y0, 'pixel_size_x': dx, 'pixel_size_y': dy, 'utm_zone': zone, 'hemisphere': hemisphere,
}
pd.DataFrame([summary]).to_csv(OUT_DIR/'00_scene_summary.csv', index=False)
pd.DataFrame({'band_1based':good_bands+1,'wavelength_nm':wl,'nodata_fraction':nodata_frac[good_bands],
              'gain':gain_good,'offset':offset_good,'model_weight':band_weight}).to_csv(OUT_DIR/'01_wavelengths_qc.csv', index=False)



In [ ]:
# ============================================================
# 4) MODEL: ABSORPTION-AWARE MASKED SPECTRAL AUTOENCODER (AAMSAE)
# ============================================================
class SpectralEncoder(nn.Module):
    def __init__(self, n_bands, latent_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(3, 32, 7, padding=3), nn.BatchNorm1d(32), nn.GELU(),
            nn.Conv1d(32, 64, 5, stride=2, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, 96, 5, stride=2, padding=2), nn.BatchNorm1d(96), nn.GELU(),
            nn.Conv1d(96, 128, 3, stride=2, padding=1), nn.BatchNorm1d(128), nn.GELU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.fc = nn.Linear(128, latent_dim)
    def forward(self, x3):
        h = self.net(x3).squeeze(-1)
        return self.fc(h)

class AAMSAE(nn.Module):
    def __init__(self, n_bands, latent_dim=32, wl_norm=None):
        super().__init__()
        self.n_bands = n_bands
        self.encoder = SpectralEncoder(n_bands, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.GELU(), nn.Dropout(0.05),
            nn.Linear(128, 256), nn.GELU(),
            nn.Linear(256, n_bands)
        )
        self.register_buffer('wl', torch.tensor(wl_norm, dtype=torch.float32).view(1,1,-1))
    def make_input(self, x, mask=None):
        if mask is None: mask = torch.zeros_like(x, dtype=torch.bool)
        xm = x.masked_fill(mask, 0.0)
        w = self.wl.expand(x.shape[0], -1, -1)
        return torch.cat([xm.unsqueeze(1), mask.float().unsqueeze(1), w], dim=1)
    def encode(self, x):
        return self.encoder(self.make_input(x, None))
    def forward(self, x, mask=None):
        z = self.encoder(self.make_input(x, mask))
        rec = self.decoder(z)
        return rec, z

bw_t = torch.tensor(band_weight, dtype=torch.float32, device=DEVICE).view(1,-1)
dv_t = torch.tensor(deriv_valid, dtype=torch.float32, device=DEVICE).view(1,-1)

def masked_loss(pred, target, mask, mode='full'):
    sq = (pred-target)**2
    if mode == 'plain':
        return sq.mean()
    # masked MSE
    denom = (mask.float()*bw_t).sum().clamp_min(1.0)
    recon = (sq * mask.float() * bw_t).sum() / denom
    if mode == 'masked':
        return recon
    # full AAMSAE = masked reconstruction + wavelength-aware derivative matching
    dp = pred[:,1:] - pred[:,:-1]
    dt = target[:,1:] - target[:,:-1]
    dmask = (mask[:,1:] | mask[:,:-1]).float() * dv_t
    dloss = (((dp-dt)**2) * dmask).sum() / dmask.sum().clamp_min(1.0)
    # small global bottleneck reconstruction term stabilizes anomaly scoring
    global_loss = (sq * bw_t).mean()
    return recon + 0.15*dloss + 0.03*global_loss


def train_model(mode, epochs):
    model = AAMSAE(len(wl), LATENT_DIM, wl_norm).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs,1), eta_min=2e-4)
    ds = TensorDataset(torch.from_numpy(X_train))
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=False)
    losses=[]
    best=np.inf; best_state=None
    for ep in range(1,epochs+1):
        model.train(); running=0.0; n=0
        for (xb,) in dl:
            xb=xb.to(DEVICE, non_blocking=True)
            if mode == 'plain':
                mask = torch.zeros_like(xb, dtype=torch.bool)
            else:
                mask = torch.rand_like(xb) < MASK_RATIO
                # guarantee at least one masked band per spectrum
                empty = ~mask.any(dim=1)
                if empty.any(): mask[empty, 0] = True
            pred,_ = model(xb, mask)
            loss = masked_loss(pred, xb, mask, mode=mode)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            running += float(loss.item())*len(xb); n += len(xb)
        scheduler.step()
        epoch_loss = running/max(n,1); losses.append(epoch_loss)
        if epoch_loss < best:
            best=epoch_loss
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        if ep==1 or ep%5==0 or ep==epochs:
            print(f'{mode:>6} epoch {ep:02d}/{epochs} loss={epoch_loss:.6f}')
    model.load_state_dict(best_state)
    model.to(DEVICE).eval()
    return model, losses

@torch.no_grad()
def encode_reconstruct(model, X, batch=4096):
    Z=[]; R=[]
    for s in range(0,len(X),batch):
        xb=torch.from_numpy(X[s:s+batch]).to(DEVICE)
        rec,z=model(xb, None)
        Z.append(z.cpu().numpy())
        R.append(rec.cpu().numpy())
    return np.vstack(Z), np.vstack(R)



In [ ]:
# ============================================================
# 5) BASELINE + ABLATION
# ============================================================
def choose_k(E_train, E_eval, method_name):
    scaler = StandardScaler().fit(E_train)
    tr=scaler.transform(E_train); ev=scaler.transform(E_eval)
    rows=[]; models={}
    # Use a capped subset for silhouette memory/time.
    eval_sel=np.arange(len(ev))
    if len(ev)>6000: eval_sel=rng.choice(len(ev),6000,replace=False)
    for k in K_RANGE:
        km=MiniBatchKMeans(n_clusters=k, random_state=SEED, n_init=10, batch_size=2048).fit(tr)
        lab=km.predict(ev)
        labs=lab[eval_sel]; e=ev[eval_sel]
        if len(np.unique(labs))<2: continue
        rows.append({'method':method_name,'k':k,
                     'silhouette':float(silhouette_score(e,labs)),
                     'davies_bouldin':float(davies_bouldin_score(e,labs)),
                     'calinski_harabasz':float(calinski_harabasz_score(e,labs))})
        models[k]=(scaler,km)
    df=pd.DataFrame(rows)
    best_k=int(df.sort_values(['silhouette','davies_bouldin'],ascending=[False,True]).iloc[0]['k'])
    return df,best_k,models[best_k]

# PCA baseline
pca_dim=min(20, len(wl), len(X_train)-1)
pca=PCA(n_components=pca_dim, random_state=SEED).fit(X_train)
P_tr=pca.transform(X_train); P_ev=pca.transform(X_eval)
kdfs=[]; summary_rows=[]
pca_kdf,pca_k,(pca_es,pca_km)=choose_k(P_tr,P_ev,'PCA+KMeans')
kdfs.append(pca_kdf)
summary_rows.append({'method':'PCA+KMeans','best_k':pca_k,
                     **pca_kdf[pca_kdf.k==pca_k][['silhouette','davies_bouldin','calinski_harabasz']].iloc[0].to_dict(),
                     'reconstruction_mse':np.nan,'cluster_stability_ARI':np.nan})

models={}; loss_history={}
variants=[]
if RUN_ABLATION:
    variants += [('plain', ABLATION_EPOCHS), ('masked', ABLATION_EPOCHS)]
variants += [('full', FULL_EPOCHS)]

for mode,epochs in variants:
    model,losses=train_model(mode,epochs)
    models[mode]=model; loss_history[mode]=losses
    Ztr,Rtr=encode_reconstruct(model,X_train,MAP_BATCH)
    Zev,Rev=encode_reconstruct(model,X_eval,MAP_BATCH)
    kdf,best_k,(es,km)=choose_k(Ztr,Zev, {'plain':'AE','masked':'MSAE','full':'AAMSAE'}[mode])
    kdfs.append(kdf)
    # Reconstruction MSE in standardized spectral space
    rec_mse=float(np.mean((Rev-X_eval)**2))
    # Clustering stability across random initializations at chosen k
    ztr_s=es.transform(Ztr); zev_s=es.transform(Zev)
    labs=[]
    for sd in [11,23,37]:
        ktmp=MiniBatchKMeans(n_clusters=best_k,random_state=sd,n_init=10,batch_size=2048).fit(ztr_s)
        labs.append(ktmp.predict(zev_s))
    ari=np.mean([adjusted_rand_score(labs[i],labs[j]) for i in range(3) for j in range(i+1,3)])
    bestrow=kdf[kdf.k==best_k].iloc[0]
    summary_rows.append({'method':{'plain':'AE','masked':'MSAE','full':'AAMSAE'}[mode], 'best_k':best_k,
                         'silhouette':float(bestrow.silhouette),'davies_bouldin':float(bestrow.davies_bouldin),
                         'calinski_harabasz':float(bestrow.calinski_harabasz),
                         'reconstruction_mse':rec_mse,'cluster_stability_ARI':float(ari)})
    if mode=='full':
        FULL_MODEL=model; FULL_ZTR=Ztr; FULL_RTR=Rtr; FULL_ZEV=Zev; FULL_REV=Rev
        FULL_K=best_k; FULL_ES=es; FULL_KM=km

kdf_all=pd.concat(kdfs,ignore_index=True)
kdf_all.to_csv(OUT_DIR/'02_k_selection_metrics.csv',index=False)
ablation_df=pd.DataFrame(summary_rows)
ablation_df.to_csv(OUT_DIR/'03_ablation_baseline_metrics.csv',index=False)
print('\nAblation/baseline summary:')
print(ablation_df.to_string(index=False))

# Training-loss figure
plt.figure(figsize=(8,5))
for mode,losses in loss_history.items():
    plt.plot(np.arange(1,len(losses)+1),losses,label=mode)
plt.xlabel('Epoch'); plt.ylabel('Training loss'); plt.title('Self-supervised training curves'); plt.legend(); plt.tight_layout()
plt.savefig(OUT_DIR/'fig_01_training_loss.png',dpi=220); plt.close()



In [ ]:
# ============================================================
# 6) FULL-SCENE AAMSAE CLUSTERING + ENSEMBLE ANOMALY MAP
# ============================================================
# Fit robust latent RX on AAMSAE training embeddings.
Ztr_s=FULL_ES.transform(FULL_ZTR); Zev_s=FULL_ES.transform(FULL_ZEV)
rx=LedoitWolf().fit(Ztr_s)
prec=rx.precision_.astype(np.float32); center=rx.location_.astype(np.float32)
centroids=FULL_KM.cluster_centers_.astype(np.float32)

# Reference score distributions on eval sample for robust normalization.
def rx_score(Zs):
    d=Zs-center
    return np.einsum('bi,ij,bj->b',d,prec,d)

def centroid_distance(Zs):
    lab=FULL_KM.predict(Zs)
    return np.linalg.norm(Zs-centroids[lab],axis=1)

rec_ref=np.mean((FULL_REV-X_eval)**2 * band_weight.reshape(1,-1),axis=1)
rx_ref=rx_score(Zev_s)
cd_ref=centroid_distance(Zev_s)

def robust_params(x):
    med=float(np.median(x)); mad=float(np.median(np.abs(x-med)))+1e-8
    return med,1.4826*mad
rec_par=robust_params(rec_ref); rx_par=robust_params(rx_ref); cd_par=robust_params(cd_ref)

def rz(x,par): return np.clip((x-par[0])/par[1],-5,15)

cluster_map=np.full((lines,samples),-1,dtype=np.int16)
anomaly_map=np.full((lines,samples),np.nan,dtype=np.float32)
recon_map=np.full((lines,samples),np.nan,dtype=np.float32)
rx_map=np.full((lines,samples),np.nan,dtype=np.float32)
cd_map=np.full((lines,samples),np.nan,dtype=np.float32)

coords_map=valid_coords[::MAP_STRIDE]
print('Full-scene inference pixels:',len(coords_map))
FULL_MODEL.eval()
with torch.no_grad():
    for s in range(0,len(coords_map),MAP_BATCH):
        cc=coords_map[s:s+MAP_BATCH]
        Xraw=read_spectra(cc); Xn=normalize_spectra(Xraw)
        xb=torch.from_numpy(Xn).to(DEVICE)
        rec,z=FULL_MODEL(xb,None)
        rec=rec.cpu().numpy(); z=z.cpu().numpy()
        zs=FULL_ES.transform(z)
        lab=FULL_KM.predict(zs)
        rerr=np.mean((rec-Xn)**2 * band_weight.reshape(1,-1),axis=1)
        rxs=rx_score(zs)
        cds=centroid_distance(zs)
        score=(rz(rerr,rec_par)+rz(rxs,rx_par)+rz(cds,cd_par))/3.0
        rr,ccol=cc[:,0],cc[:,1]
        cluster_map[rr,ccol]=lab.astype(np.int16)
        anomaly_map[rr,ccol]=score.astype(np.float32)
        recon_map[rr,ccol]=rerr.astype(np.float32)
        rx_map[rr,ccol]=rxs.astype(np.float32)
        cd_map[rr,ccol]=cds.astype(np.float32)
        if s==0 or (s//MAP_BATCH)%50==0:
            print(f'  processed {min(s+len(cc),len(coords_map))}/{len(coords_map)}')

# Optional categorical 3x3 spatial consensus map (not used as ground truth).
try:
    from scipy.ndimage import uniform_filter
    scores=[]
    for k in range(FULL_K):
        scores.append(uniform_filter((cluster_map==k).astype(np.float32),size=3,mode='nearest'))
    smooth=np.argmax(np.stack(scores,axis=0),axis=0).astype(np.int16)
    cluster_map_smooth=np.where(cluster_map>=0,smooth,-1).astype(np.int16)
except Exception:
    cluster_map_smooth=cluster_map.copy()


def spatial_coherence(lbl):
    m=lbl>=0
    a=(lbl[:,:-1]==lbl[:,1:]) & m[:,:-1] & m[:,1:]
    b=(lbl[:-1,:]==lbl[1:,:]) & m[:-1,:] & m[1:,:]
    den=(m[:,:-1]&m[:,1:]).sum()+(m[:-1,:]&m[1:,:]).sum()
    return float((a.sum()+b.sum())/max(den,1))

coh_raw=spatial_coherence(cluster_map); coh_smooth=spatial_coherence(cluster_map_smooth)
pd.DataFrame([{'method':'AAMSAE raw','spatial_neighbor_agreement':coh_raw},
              {'method':'AAMSAE 3x3 consensus','spatial_neighbor_agreement':coh_smooth}]).to_csv(OUT_DIR/'04_spatial_coherence.csv',index=False)

np.save(OUT_DIR/'cluster_map_raw.npy',cluster_map)
np.save(OUT_DIR/'cluster_map_consensus.npy',cluster_map_smooth)
np.save(OUT_DIR/'anomaly_score.npy',anomaly_map)
np.save(OUT_DIR/'reconstruction_error.npy',recon_map)
np.save(OUT_DIR/'latent_rx_score.npy',rx_map)
np.save(OUT_DIR/'centroid_distance.npy',cd_map)



In [ ]:
# ============================================================
# 7) PAPER FIGURES
# ============================================================
def percentile_stretch(x,mask,p1=2,p2=98):
    vals=x[mask & np.isfinite(x)]
    if len(vals)==0: return np.zeros_like(x,dtype=np.float32)
    lo,hi=np.percentile(vals,[p1,p2]);
    return np.clip((x-lo)/(hi-lo+1e-8),0,1)

# RGB proxy from nearest EnMAP bands.
rgb=[]
for target in [650,560,470]:
    j=int(np.argmin(np.abs(wavelengths-target)))
    arr=np.asarray(cube[j],dtype=np.float32)*gains[j]+offsets[j]
    rgb.append(percentile_stretch(arr,valid_mask))
rgb=np.stack(rgb,axis=-1)
rgb[~valid_mask]=0
plt.figure(figsize=(8,8)); plt.imshow(rgb); plt.axis('off'); plt.title('EnMAP L2A RGB proxy'); plt.tight_layout()
plt.savefig(OUT_DIR/'fig_02_rgb_proxy.png',dpi=250,bbox_inches='tight'); plt.close()

# Cluster maps
for name,cm,title in [('raw',cluster_map,'AAMSAE spectral units (raw)'),('consensus',cluster_map_smooth,'AAMSAE spectral units (3×3 consensus)')]:
    show=np.ma.masked_where(cm<0,cm)
    plt.figure(figsize=(8,8)); plt.imshow(show,cmap='tab10',interpolation='nearest'); plt.axis('off'); plt.title(title)
    cb=plt.colorbar(fraction=0.046,pad=0.04); cb.set_label('Spectral unit ID')
    plt.tight_layout(); plt.savefig(OUT_DIR/f'fig_03_cluster_map_{name}.png',dpi=250,bbox_inches='tight'); plt.close()

# Anomaly map
show=np.ma.masked_invalid(anomaly_map)
vmax=np.nanpercentile(anomaly_map,99.5)
plt.figure(figsize=(8,8)); plt.imshow(show,cmap='magma',vmin=np.nanpercentile(anomaly_map,5),vmax=vmax); plt.axis('off'); plt.title('AAMSAE ensemble spectral anomaly score')
cb=plt.colorbar(fraction=0.046,pad=0.04); cb.set_label('Robust anomaly score')
plt.tight_layout(); plt.savefig(OUT_DIR/'fig_04_anomaly_map.png',dpi=250,bbox_inches='tight'); plt.close()

# Cluster mean spectra from evaluation/train sample, using full-model assignments.
all_raw=np.vstack([X_train_raw,X_eval_raw])
all_norm=np.vstack([X_train,X_eval])
all_z=np.vstack([FULL_ZTR,FULL_ZEV])
all_labels=FULL_KM.predict(FULL_ES.transform(all_z))
cluster_spec={'wavelength_nm':wl}
plt.figure(figsize=(10,6))
for k in range(FULL_K):
    idx=np.where(all_labels==k)[0]
    if len(idx)==0: continue
    if len(idx)>5000: idx=rng.choice(idx,5000,replace=False)
    mean_spec=np.mean(all_raw[idx],axis=0)
    cluster_spec[f'unit_{k}_mean_reflectance']=mean_spec
    plt.plot(wl,mean_spec,label=f'Unit {k} (n={len(idx)})')
plt.xlabel('Wavelength (nm)'); plt.ylabel('Surface reflectance'); plt.title('Mean spectra of discovered AAMSAE units'); plt.legend(fontsize=8,ncol=2); plt.tight_layout()
plt.savefig(OUT_DIR/'fig_05_cluster_mean_spectra.png',dpi=250); plt.close()
pd.DataFrame(cluster_spec).to_csv(OUT_DIR/'05_cluster_mean_spectra.csv',index=False)

# 2D latent visualization using PCA on latent embedding.
lat2=PCA(n_components=2,random_state=SEED).fit_transform(FULL_ZEV)
lab2=FULL_KM.predict(FULL_ES.transform(FULL_ZEV))
sel2=np.arange(len(lat2))
if len(sel2)>8000: sel2=rng.choice(len(sel2),8000,replace=False)
plt.figure(figsize=(7,6)); plt.scatter(lat2[sel2,0],lat2[sel2,1],c=lab2[sel2],s=5,cmap='tab10',alpha=.65); plt.xlabel('Latent PC1'); plt.ylabel('Latent PC2'); plt.title('AAMSAE latent spectral space'); plt.tight_layout()
plt.savefig(OUT_DIR/'fig_06_latent_space.png',dpi=250); plt.close()



In [ ]:
# ============================================================
# 8) TOP ANOMALIES TABLE + COORDINATES
# ============================================================
score_flat=anomaly_map[valid_mask & np.isfinite(anomaly_map)]
coord_valid=np.argwhere(valid_mask & np.isfinite(anomaly_map))
order=np.argsort(score_flat)[::-1]
selected=[]
min_sep_px=8
for idx in order:
    r,c=coord_valid[idx]
    if all((r-r0)**2+(c-c0)**2 >= min_sep_px**2 for r0,c0 in selected):
        selected.append((int(r),int(c)))
    if len(selected)>=TOP_ANOMALIES: break

rows=[]
# optional WGS84 conversion
transformer=None
if zone is not None:
    try:
        from pyproj import Transformer
        epsg=32600+zone if str(hemisphere).lower().startswith('n') else 32700+zone
        transformer=Transformer.from_crs(f'EPSG:{epsg}','EPSG:4326',always_xy=True)
    except Exception:
        transformer=None
for rank,(r,c) in enumerate(selected,1):
    x = (x0 + c*dx) if x0 is not None else np.nan
    y = (y0 - r*dy) if y0 is not None else np.nan
    lon=lat=np.nan
    if transformer is not None:
        lon,lat=transformer.transform(x,y)
    rows.append({'rank':rank,'row':r,'col':c,'anomaly_score':float(anomaly_map[r,c]),
                 'cluster_raw':int(cluster_map[r,c]),'cluster_consensus':int(cluster_map_smooth[r,c]),
                 'utm_x':x,'utm_y':y,'longitude':lon,'latitude':lat,
                 'reconstruction_error':float(recon_map[r,c]),'latent_rx_score':float(rx_map[r,c])})
pd.DataFrame(rows).to_csv(OUT_DIR/'06_top_anomaly_candidates.csv',index=False)




In [ ]:
# ============================================================
# 9) PROFESSIONAL PAPER-READY RESULTS PACKAGE
# ============================================================
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.ticker import MaxNLocator

PUB_DIR = OUT_DIR / 'paper_ready_figures'
TABLE_DIR = OUT_DIR / 'tables'
GEO_DIR = OUT_DIR / 'geospatial'
PUB_DIR.mkdir(exist_ok=True)
TABLE_DIR.mkdir(exist_ok=True)
GEO_DIR.mkdir(exist_ok=True)

def save_pubfig(fig, stem, dpi=350):
    """Save every paper figure in high-res PNG + vector PDF."""
    fig.savefig(PUB_DIR / f'{stem}.png', dpi=dpi, bbox_inches='tight')
    fig.savefig(PUB_DIR / f'{stem}.pdf', bbox_inches='tight')
    plt.close(fig)

# ------------------------------------------------------------
# 9.1 Scientific model ranking / safeguard
# ------------------------------------------------------------
rank_df = ablation_df.copy()
rank_df['rank_silhouette'] = rank_df['silhouette'].rank(ascending=False, method='min')
rank_df['rank_davies_bouldin'] = rank_df['davies_bouldin'].rank(ascending=True, method='min')
rank_df['rank_calinski_harabasz'] = rank_df['calinski_harabasz'].rank(ascending=False, method='min')
rank_df['rank_stability'] = rank_df['cluster_stability_ARI'].rank(ascending=False, method='min', na_option='bottom')
rank_df['cluster_rank_mean'] = rank_df[
    ['rank_silhouette','rank_davies_bouldin','rank_calinski_harabasz','rank_stability']
].mean(axis=1)
rank_df['cluster_rank_final'] = rank_df['cluster_rank_mean'].rank(ascending=True, method='min')
rank_df = rank_df.sort_values(['cluster_rank_final','silhouette'], ascending=[True,False])
rank_df.to_csv(TABLE_DIR/'07_method_ranking.csv', index=False)

best_cluster_method = str(rank_df.iloc[0]['method'])
aamsae_row = ablation_df[ablation_df.method=='AAMSAE'].iloc[0]
msae_row = ablation_df[ablation_df.method=='MSAE'].iloc[0] if (ablation_df.method=='MSAE').any() else None
decision_lines = [
    'SCIENTIFIC DECISION CHECK',
    '=========================',
    f'Best cluster-quality method in this run: {best_cluster_method}',
    f'AAMSAE silhouette: {aamsae_row.silhouette:.6f}',
    f'AAMSAE Davies-Bouldin: {aamsae_row.davies_bouldin:.6f}',
    f'AAMSAE Calinski-Harabasz: {aamsae_row.calinski_harabasz:.3f}',
    f'AAMSAE stability ARI: {aamsae_row.cluster_stability_ARI:.6f}',
]
if msae_row is not None:
    decision_lines += [
        f'MSAE silhouette: {msae_row.silhouette:.6f}',
        f'MSAE stability ARI: {msae_row.cluster_stability_ARI:.6f}',
        '',
        'IMPORTANT:',
        'Do not claim that AAMSAE is superior for clustering unless the metrics support that statement.',
        'If AAMSAE is retained, frame its role around spectral reconstruction/anomaly discovery and validate anomalies independently.',
    ]
(TABLE_DIR/'SCIENTIFIC_DECISION.txt').write_text('\n'.join(decision_lines))

# ------------------------------------------------------------
# 9.2 Cluster spatial statistics
# ------------------------------------------------------------
pixel_area_km2 = (abs(dx*dy) / 1e6) if (dx is not None and dy is not None) else np.nan
cluster_stats = []
for k in range(FULL_K):
    m = cluster_map_smooth == k
    vals = anomaly_map[m & np.isfinite(anomaly_map)]
    n = int(m.sum())
    cluster_stats.append({
        'spectral_unit': int(k),
        'pixel_count': n,
        'percentage_valid_area': 100.0*n/max(int(valid_mask.sum()),1),
        'area_km2': n*pixel_area_km2 if np.isfinite(pixel_area_km2) else np.nan,
        'mean_anomaly_score': float(np.nanmean(vals)) if len(vals) else np.nan,
        'median_anomaly_score': float(np.nanmedian(vals)) if len(vals) else np.nan,
        'p95_anomaly_score': float(np.nanpercentile(vals,95)) if len(vals) else np.nan,
        'max_anomaly_score': float(np.nanmax(vals)) if len(vals) else np.nan,
    })
cluster_stats_df = pd.DataFrame(cluster_stats)
cluster_stats_df.to_csv(TABLE_DIR/'08_cluster_spatial_statistics.csv', index=False)

# ------------------------------------------------------------
# 9.3 Anomaly threshold sensitivity table
# ------------------------------------------------------------
valid_scores = anomaly_map[np.isfinite(anomaly_map)]
thr_rows=[]
for p in [90, 95, 97.5, 99, 99.5, 99.9]:
    th=float(np.nanpercentile(valid_scores,p))
    count=int(np.sum(valid_scores>=th))
    thr_rows.append({
        'percentile':p, 'threshold':th, 'pixels_flagged':count,
        'fraction_percent':100*count/max(len(valid_scores),1),
        'area_km2':count*pixel_area_km2 if np.isfinite(pixel_area_km2) else np.nan
    })
threshold_df=pd.DataFrame(thr_rows)
threshold_df.to_csv(TABLE_DIR/'09_anomaly_threshold_sensitivity.csv',index=False)

# ------------------------------------------------------------
# 9.4 KMeans seed robustness for AAMSAE embedding
# ------------------------------------------------------------
robust_seeds=[3,7,11,17,23,31,37,43,53,67]
seed_labels=[]
seed_rows=[]
eval_metric_idx=np.arange(len(Zev_s))
if len(eval_metric_idx)>5000:
    eval_metric_idx=rng.choice(len(eval_metric_idx),5000,replace=False)
for sd in robust_seeds:
    km_tmp=MiniBatchKMeans(n_clusters=FULL_K,random_state=sd,n_init=10,batch_size=2048).fit(Ztr_s)
    lab_tmp=km_tmp.predict(Zev_s)
    seed_labels.append(lab_tmp)
    em=Zev_s[eval_metric_idx]; lm=lab_tmp[eval_metric_idx]
    seed_rows.append({
        'seed':sd,
        'silhouette':float(silhouette_score(em,lm)),
        'davies_bouldin':float(davies_bouldin_score(em,lm)),
        'calinski_harabasz':float(calinski_harabasz_score(em,lm))
    })
seed_robust_df=pd.DataFrame(seed_rows)
ari_matrix=np.eye(len(robust_seeds),dtype=np.float32)
for i in range(len(robust_seeds)):
    for j in range(i+1,len(robust_seeds)):
        a=adjusted_rand_score(seed_labels[i],seed_labels[j])
        ari_matrix[i,j]=ari_matrix[j,i]=a
pairwise_ari=ari_matrix[np.triu_indices_from(ari_matrix,k=1)]
seed_summary_df=pd.DataFrame([{
    'n_seeds':len(robust_seeds),
    'silhouette_mean':seed_robust_df.silhouette.mean(),
    'silhouette_std':seed_robust_df.silhouette.std(ddof=1),
    'davies_bouldin_mean':seed_robust_df.davies_bouldin.mean(),
    'davies_bouldin_std':seed_robust_df.davies_bouldin.std(ddof=1),
    'calinski_harabasz_mean':seed_robust_df.calinski_harabasz.mean(),
    'calinski_harabasz_std':seed_robust_df.calinski_harabasz.std(ddof=1),
    'pairwise_ARI_mean':float(pairwise_ari.mean()),
    'pairwise_ARI_std':float(pairwise_ari.std(ddof=1)),
    'pairwise_ARI_min':float(pairwise_ari.min()),
    'pairwise_ARI_max':float(pairwise_ari.max()),
}])
seed_robust_df.to_csv(TABLE_DIR/'10_seed_robustness_runs.csv',index=False)
seed_summary_df.to_csv(TABLE_DIR/'10_seed_robustness_summary.csv',index=False)
pd.DataFrame(ari_matrix,index=robust_seeds,columns=robust_seeds).to_csv(TABLE_DIR/'10_seed_ARI_matrix.csv')

# ------------------------------------------------------------
# 9.5 Lightweight bootstrap confidence intervals
# ------------------------------------------------------------
boot_rows=[]
boot_n=min(1000,len(Zev_s))
n_boot=30
for b in range(n_boot):
    idx=rng.choice(len(Zev_s),boot_n,replace=True)
    e=Zev_s[idx]
    lab=FULL_KM.predict(e)
    if len(np.unique(lab))<2:
        continue
    # silhouette on <=1000 points is tractable and gives an uncertainty diagnostic.
    boot_rows.append({
        'bootstrap':b+1,
        'silhouette':float(silhouette_score(e,lab)),
        'davies_bouldin':float(davies_bouldin_score(e,lab)),
        'calinski_harabasz':float(calinski_harabasz_score(e,lab)),
    })
boot_df=pd.DataFrame(boot_rows)
boot_df.to_csv(TABLE_DIR/'11_bootstrap_metrics.csv',index=False)
ci_rows=[]
for c in ['silhouette','davies_bouldin','calinski_harabasz']:
    arr=boot_df[c].to_numpy()
    ci_rows.append({
        'metric':c,'mean':float(np.mean(arr)),'std':float(np.std(arr,ddof=1)),
        'ci95_low':float(np.percentile(arr,2.5)),'ci95_high':float(np.percentile(arr,97.5))
    })
bootstrap_ci_df=pd.DataFrame(ci_rows)
bootstrap_ci_df.to_csv(TABLE_DIR/'11_bootstrap_95CI.csv',index=False)

# ------------------------------------------------------------
# 9.6 Candidate spectral absorption-feature table (descriptive only)
# ------------------------------------------------------------
try:
    from scipy.signal import savgol_filter, find_peaks
    spectral_features=[]
    for k in range(FULL_K):
        col=f'unit_{k}_mean_reflectance'
        spec=np.asarray(cluster_spec[col],dtype=float)
        win=min(17, len(spec)//2*2-1)
        win=max(win,5)
        smooth_spec=savgol_filter(spec,win,2)
        inv=-smooth_spec
        prom=max(0.002,0.02*np.nanstd(smooth_spec))
        peaks,props=find_peaks(inv,prominence=prom,distance=3)
        # Keep only the strongest 10 candidate minima per unit.
        if len(peaks):
            orderp=np.argsort(props['prominences'])[::-1][:10]
            for ix in orderp:
                pidx=int(peaks[ix])
                spectral_features.append({
                    'spectral_unit':k,
                    'candidate_absorption_nm':float(wl[pidx]),
                    'mean_reflectance':float(spec[pidx]),
                    'prominence':float(props['prominences'][ix]),
                    'note':'Candidate spectral minimum only; mineral identity requires independent validation.'
                })
    spectral_features_df=pd.DataFrame(spectral_features)
except Exception:
    spectral_features_df=pd.DataFrame(columns=['spectral_unit','candidate_absorption_nm','mean_reflectance','prominence','note'])
spectral_features_df.to_csv(TABLE_DIR/'12_candidate_absorption_features.csv',index=False)

# ------------------------------------------------------------
# 9.7 Component-level anomaly statistics
# ------------------------------------------------------------
component_sample=np.argwhere(valid_mask & np.isfinite(anomaly_map))
if len(component_sample)>50000:
    component_sample=component_sample[rng.choice(len(component_sample),50000,replace=False)]
rrs,ccs=component_sample[:,0],component_sample[:,1]
component_df=pd.DataFrame({
    'reconstruction_error':recon_map[rrs,ccs],
    'latent_rx_score':rx_map[rrs,ccs],
    'centroid_distance':cd_map[rrs,ccs],
    'ensemble_anomaly':anomaly_map[rrs,ccs],
    'spectral_unit':cluster_map_smooth[rrs,ccs]
})
component_df.corr(numeric_only=True).to_csv(TABLE_DIR/'13_anomaly_component_correlations.csv')
component_df.describe(percentiles=[.5,.9,.95,.99,.995]).T.to_csv(TABLE_DIR/'13_anomaly_component_summary.csv')

# ------------------------------------------------------------
# 9.8 Professional figures
# ------------------------------------------------------------

# Figure 07 — K selection metrics
fig,axes=plt.subplots(1,3,figsize=(15,4.6))
metric_specs=[
    ('silhouette','Silhouette score','Higher is better'),
    ('davies_bouldin','Davies–Bouldin index','Lower is better'),
    ('calinski_harabasz','Calinski–Harabasz score','Higher is better')
]
for ax,(metric,ylabel,note) in zip(axes,metric_specs):
    for method,g in kdf_all.groupby('method'):
        g=g.sort_values('k')
        ax.plot(g.k,g[metric],marker='o',label=method)
    ax.set_xlabel('Number of clusters (K)')
    ax.set_ylabel(ylabel)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=.2)
    ax.text(.02,.98,note,transform=ax.transAxes,va='top',fontsize=8)
axes[0].legend(frameon=False,fontsize=8)
fig.suptitle('Internal cluster-quality analysis across K')
fig.tight_layout()
save_pubfig(fig,'fig_07_k_selection_metrics')

# Figure 08 — Ablation/baseline comparison
fig,axes=plt.subplots(2,2,figsize=(11,8))
plot_metrics=[
    ('silhouette','Silhouette ↑'),
    ('davies_bouldin','Davies–Bouldin ↓'),
    ('calinski_harabasz','Calinski–Harabasz ↑'),
    ('cluster_stability_ARI','Clustering stability ARI ↑')
]
for ax,(metric,title) in zip(axes.ravel(),plot_metrics):
    d=ablation_df[['method',metric]].dropna()
    ax.bar(d.method,d[metric])
    ax.set_title(title)
    ax.tick_params(axis='x',rotation=25)
    ax.grid(axis='y',alpha=.2)
fig.suptitle('Baseline and ablation comparison')
fig.tight_layout()
save_pubfig(fig,'fig_08_ablation_comparison')

# Figure 09 — Reconstruction vs clusterability tradeoff
d=ablation_df.dropna(subset=['reconstruction_mse']).copy()
fig,ax=plt.subplots(figsize=(7,5.6))
ax.scatter(d.reconstruction_mse,d.silhouette,s=90)
for _,r in d.iterrows():
    ax.annotate(r.method,(r.reconstruction_mse,r.silhouette),xytext=(5,5),textcoords='offset points')
ax.set_xlabel('Reconstruction MSE ↓')
ax.set_ylabel('Silhouette score ↑')
ax.set_title('Reconstruction–clusterability trade-off')
ax.grid(alpha=.2)
save_pubfig(fig,'fig_09_reconstruction_clusterability_tradeoff')

# Figure 10 — Spectral-unit area distribution
fig,axes=plt.subplots(1,2,figsize=(11,4.5))
axes[0].bar(cluster_stats_df.spectral_unit.astype(str),cluster_stats_df.percentage_valid_area)
axes[0].set_xlabel('Spectral unit'); axes[0].set_ylabel('Valid scene area (%)')
axes[0].set_title('Spatial prevalence of discovered units'); axes[0].grid(axis='y',alpha=.2)
axes[1].bar(cluster_stats_df.spectral_unit.astype(str),cluster_stats_df.mean_anomaly_score)
axes[1].set_xlabel('Spectral unit'); axes[1].set_ylabel('Mean anomaly score')
axes[1].set_title('Mean anomaly score by unit'); axes[1].grid(axis='y',alpha=.2)
fig.tight_layout()
save_pubfig(fig,'fig_10_unit_area_and_anomaly_statistics')

# Figure 11 — Mean spectra with uncertainty envelopes
fig,ax=plt.subplots(figsize=(11,6))
for k in range(FULL_K):
    idx=np.where(all_labels==k)[0]
    if len(idx)==0: continue
    if len(idx)>6000: idx=rng.choice(idx,6000,replace=False)
    s=all_raw[idx]
    mean=s.mean(axis=0)
    std=s.std(axis=0)
    line=ax.plot(wl,mean,label=f'Unit {k}')[0]
    ax.fill_between(wl,mean-std,mean+std,alpha=.12)
ax.set_xlabel('Wavelength (nm)'); ax.set_ylabel('Surface reflectance')
ax.set_title('Spectral signatures of discovered units (mean ± 1 SD)')
ax.legend(ncol=2,frameon=False); ax.grid(alpha=.15)
fig.tight_layout()
save_pubfig(fig,'fig_11_unit_spectra_uncertainty')

# Figure 12 — Derivative spectra
fig,ax=plt.subplots(figsize=(11,5.6))
for k in range(FULL_K):
    spec=np.asarray(cluster_spec[f'unit_{k}_mean_reflectance'])
    deriv=np.gradient(spec,wl)
    deriv[np.r_[False,np.diff(wl)>25]]=np.nan
    ax.plot(wl,deriv,label=f'Unit {k}')
ax.axhline(0,lw=.8)
ax.set_xlabel('Wavelength (nm)'); ax.set_ylabel('dReflectance / dλ')
ax.set_title('First-derivative spectral signatures')
ax.legend(ncol=2,frameon=False); ax.grid(alpha=.15)
fig.tight_layout()
save_pubfig(fig,'fig_12_unit_derivative_spectra')

# Figure 13 — Band QC and model weights
fig,axes=plt.subplots(2,1,figsize=(12,6),sharex=True)
axes[0].plot(wl,nodata_frac[good_bands])
axes[0].set_ylabel('NoData fraction')
axes[0].set_title('Band-level quality control')
axes[0].grid(alpha=.15)
axes[1].plot(wl,band_weight)
axes[1].set_xlabel('Wavelength (nm)'); axes[1].set_ylabel('Model weight')
axes[1].set_title('Spectral weighting used by AAMSAE')
axes[1].grid(alpha=.15)
fig.tight_layout()
save_pubfig(fig,'fig_13_band_qc_and_weights')

# Figure 14 — Anomaly score distribution and threshold sensitivity
fig,axes=plt.subplots(1,2,figsize=(12,4.8))
axes[0].hist(valid_scores,bins=120)
for p in [95,99,99.5]:
    th=float(np.percentile(valid_scores,p))
    axes[0].axvline(th,ls='--',label=f'P{p}: {th:.2f}')
axes[0].set_xlabel('Ensemble anomaly score'); axes[0].set_ylabel('Pixel count')
axes[0].set_title('Anomaly-score distribution'); axes[0].legend(frameon=False,fontsize=8)
axes[1].plot(threshold_df.percentile,threshold_df.pixels_flagged,marker='o')
axes[1].set_xlabel('Percentile threshold'); axes[1].set_ylabel('Flagged pixels')
axes[1].set_title('Threshold sensitivity'); axes[1].grid(alpha=.2)
fig.tight_layout()
save_pubfig(fig,'fig_14_anomaly_distribution_and_thresholds')

# Figure 15 — RGB + anomaly overlay
anom_norm=np.zeros_like(anomaly_map,dtype=np.float32)
p95,p995=np.nanpercentile(valid_scores,[95,99.5])
anom_norm=np.clip((anomaly_map-p95)/(p995-p95+1e-8),0,1)
fig,ax=plt.subplots(figsize=(9,9))
ax.imshow(rgb)
im=ax.imshow(np.ma.masked_invalid(anomaly_map),cmap='magma',vmin=p95,vmax=p995,alpha=np.nan_to_num(anom_norm)*0.82)
ax.set_axis_off(); ax.set_title('High-anomaly pixels over EnMAP RGB context')
cb=fig.colorbar(im,ax=ax,fraction=.046,pad=.03); cb.set_label('Anomaly score')
fig.tight_layout()
save_pubfig(fig,'fig_15_rgb_anomaly_overlay')

# Figure 16 — Top anomaly candidates map
top_df=pd.DataFrame(rows)
fig,ax=plt.subplots(figsize=(9,9))
im=ax.imshow(np.ma.masked_invalid(anomaly_map),cmap='magma',vmin=np.nanpercentile(valid_scores,5),vmax=p995)
show_top=top_df.head(20)
ax.scatter(show_top['col'],show_top['row'],facecolors='none',edgecolors='white',s=70,linewidths=1.2)
for _,r in show_top.iterrows():
    ax.text(r['col']+5,r['row']-5,str(int(r['rank'])),fontsize=7)
ax.set_axis_off(); ax.set_title('Top 20 spatially separated anomaly candidates')
cb=fig.colorbar(im,ax=ax,fraction=.046,pad=.03); cb.set_label('Anomaly score')
fig.tight_layout()
save_pubfig(fig,'fig_16_top20_anomaly_candidates')

# Figure 17 — Candidate zoom panels on RGB
nzoom=min(12,len(top_df))
cols=4; rows_n=int(np.ceil(nzoom/cols))
fig,axes=plt.subplots(rows_n,cols,figsize=(13,3.3*rows_n))
axes=np.atleast_1d(axes).ravel()
rad=45
for i,ax in enumerate(axes):
    if i>=nzoom:
        ax.axis('off'); continue
    r=int(top_df.iloc[i].row); c=int(top_df.iloc[i].col)
    r0=max(0,r-rad); r1=min(lines,r+rad+1); c0=max(0,c-rad); c1=min(samples,c+rad+1)
    ax.imshow(rgb[r0:r1,c0:c1])
    ax.scatter([c-c0],[r-r0],marker='+',s=80)
    ax.set_title(f"Rank {int(top_df.iloc[i]['rank'])} | score={top_df.iloc[i]['anomaly_score']:.2f}",fontsize=9)
    ax.axis('off')
fig.suptitle('RGB context around highest-ranked anomaly candidates')
fig.tight_layout()
save_pubfig(fig,'fig_17_top_anomaly_rgb_zooms')

# Figure 18 — Spectra of top anomalies against their spectral-unit mean
nplot=min(8,len(top_df))
fig,axes=plt.subplots(2,4,figsize=(16,7),sharex=True)
axes=axes.ravel()
cand_coords=np.array([[int(top_df.iloc[i].row),int(top_df.iloc[i].col)] for i in range(nplot)])
cand_specs=read_spectra(cand_coords)
for i,ax in enumerate(axes):
    if i>=nplot:
        ax.axis('off'); continue
    k=int(top_df.iloc[i].cluster_consensus)
    ax.plot(wl,cand_specs[i],label='Candidate')
    if f'unit_{k}_mean_reflectance' in cluster_spec:
        ax.plot(wl,cluster_spec[f'unit_{k}_mean_reflectance'],label=f'Unit {k} mean')
    ax.set_title(f"Rank {int(top_df.iloc[i]['rank'])} | Unit {k}",fontsize=9)
    ax.grid(alpha=.15)
    if i%4==0: ax.set_ylabel('Reflectance')
    if i>=4: ax.set_xlabel('Wavelength (nm)')
axes[0].legend(frameon=False,fontsize=8)
fig.suptitle('Top anomaly spectra versus assigned spectral-unit mean')
fig.tight_layout()
save_pubfig(fig,'fig_18_top_anomaly_spectra')

# Figure 19 — Anomaly component relationship
sample_plot=component_df
if len(sample_plot)>12000:
    sample_plot=sample_plot.sample(12000,random_state=SEED)
fig,axes=plt.subplots(1,3,figsize=(15,4.5))
pairs=[
    ('reconstruction_error','latent_rx_score'),
    ('reconstruction_error','centroid_distance'),
    ('latent_rx_score','centroid_distance')
]
for ax,(xv,yv) in zip(axes,pairs):
    sc=ax.scatter(sample_plot[xv],sample_plot[yv],c=sample_plot['ensemble_anomaly'],s=5,alpha=.35,cmap='viridis')
    ax.set_xlabel(xv.replace('_',' ')); ax.set_ylabel(yv.replace('_',' '))
    ax.grid(alpha=.15)
fig.colorbar(sc,ax=axes.ravel().tolist(),label='Ensemble anomaly')
fig.suptitle('Relationship among anomaly-score components')
fig.subplots_adjust(wspace=.28,top=.86)
save_pubfig(fig,'fig_19_anomaly_component_relationships')

# Figure 20 — Seed robustness matrix
fig,axes=plt.subplots(1,2,figsize=(12,5))
axes[0].errorbar(['Silhouette','Davies–Bouldin'],
                 [seed_robust_df.silhouette.mean(),seed_robust_df.davies_bouldin.mean()],
                 yerr=[seed_robust_df.silhouette.std(ddof=1),seed_robust_df.davies_bouldin.std(ddof=1)],
                 fmt='o',capsize=5)
axes[0].set_title('AAMSAE KMeans robustness across seeds')
axes[0].grid(axis='y',alpha=.2)
im=axes[1].imshow(ari_matrix,vmin=0,vmax=1,cmap='viridis')
axes[1].set_xticks(range(len(robust_seeds)),robust_seeds,rotation=45)
axes[1].set_yticks(range(len(robust_seeds)),robust_seeds)
axes[1].set_xlabel('Seed'); axes[1].set_ylabel('Seed'); axes[1].set_title('Pairwise ARI stability')
fig.colorbar(im,ax=axes[1],fraction=.046,pad=.04)
fig.tight_layout()
save_pubfig(fig,'fig_20_seed_robustness')

# Figure 21 — Spatial consensus diagnostic
disagree=(cluster_map>=0)&(cluster_map_smooth>=0)&(cluster_map!=cluster_map_smooth)
fig,axes=plt.subplots(1,3,figsize=(15,5))
axes[0].imshow(np.ma.masked_where(cluster_map<0,cluster_map),cmap='tab10',interpolation='nearest'); axes[0].set_title(f'Raw\nneighbor agreement={coh_raw:.3f}')
axes[1].imshow(np.ma.masked_where(cluster_map_smooth<0,cluster_map_smooth),cmap='tab10',interpolation='nearest'); axes[1].set_title(f'3×3 consensus\nneighbor agreement={coh_smooth:.3f}')
axes[2].imshow(np.ma.masked_where(~valid_mask,disagree),cmap='gray'); axes[2].set_title(f'Changed pixels\n{100*disagree.sum()/max(valid_mask.sum(),1):.2f}% of valid scene')
for ax in axes: ax.axis('off')
fig.suptitle('Effect of spatial consensus refinement')
fig.tight_layout()
save_pubfig(fig,'fig_21_spatial_consensus_diagnostic')

# Figure 22 — Method workflow diagram
fig,ax=plt.subplots(figsize=(14,3.5))
ax.set_xlim(0,14); ax.set_ylim(0,3.5); ax.axis('off')
stages=[
    (0.3,'EnMAP L2A\n224 bands'),
    (2.5,'Automatic QC\n219 usable bands'),
    (4.7,'Robust spectral\nnormalization'),
    (6.9,'AAMSAE\nmasked SSL'),
    (9.1,'Latent clustering\n+ K selection'),
    (11.3,'Ensemble anomaly\nscoring'),
]
for x,label in stages:
    box=FancyBboxPatch((x,1.25),1.8,.9,boxstyle='round,pad=0.03',fill=False,linewidth=1.5)
    ax.add_patch(box); ax.text(x+.9,1.7,label,ha='center',va='center',fontsize=9)
for i in range(len(stages)-1):
    x1=stages[i][0]+1.8; x2=stages[i+1][0]
    ax.add_patch(FancyArrowPatch((x1+.05,1.7),(x2-.05,1.7),arrowstyle='->',mutation_scale=14,linewidth=1.2))
ax.text(12.55,2.55,'Outputs',ha='center',fontsize=9)
ax.text(12.55,.72,'Spectral units • anomaly candidates\nGeoTIFFs • tables • validation targets',ha='center',fontsize=9)
ax.set_title('End-to-end label-free EnMAP analysis workflow')
fig.tight_layout()
save_pubfig(fig,'fig_22_method_workflow')

# Figure 23 — One-page paper summary
fig,axes=plt.subplots(2,3,figsize=(16,10))
axes[0,0].imshow(rgb); axes[0,0].set_title('EnMAP RGB'); axes[0,0].axis('off')
axes[0,1].imshow(np.ma.masked_where(cluster_map_smooth<0,cluster_map_smooth),cmap='tab10'); axes[0,1].set_title('Spectral units'); axes[0,1].axis('off')
axes[0,2].imshow(np.ma.masked_invalid(anomaly_map),cmap='magma',vmax=p995); axes[0,2].set_title('Anomaly score'); axes[0,2].axis('off')
for k in range(FULL_K):
    axes[1,0].plot(wl,cluster_spec[f'unit_{k}_mean_reflectance'],label=f'U{k}')
axes[1,0].set_title('Mean spectra'); axes[1,0].set_xlabel('Wavelength (nm)'); axes[1,0].set_ylabel('Reflectance'); axes[1,0].legend(frameon=False,fontsize=7)
d=ablation_df.dropna(subset=['silhouette'])
axes[1,1].bar(d.method,d.silhouette); axes[1,1].set_title('Silhouette comparison'); axes[1,1].tick_params(axis='x',rotation=25)
axes[1,2].hist(valid_scores,bins=80); axes[1,2].axvline(p995,ls='--'); axes[1,2].set_title('Anomaly-score distribution'); axes[1,2].set_xlabel('Score')
fig.suptitle('EnMAP AAMSAE analysis — publication summary panel')
fig.tight_layout()
save_pubfig(fig,'fig_23_paper_summary_panel')

# ------------------------------------------------------------
# 9.9 Geospatial raster exports (GeoTIFF)
# ------------------------------------------------------------
geotiff_status=[]
try:
    import rasterio
    from rasterio.transform import from_origin
    if all(v is not None for v in [x0,y0,dx,dy,zone]):
        transform=from_origin(x0,y0,abs(dx),abs(dy))
        epsg=(32600+zone) if str(hemisphere).lower().startswith('n') else (32700+zone)
        crs=f'EPSG:{epsg}'
        def write_tif(path,arr,dtype,nodata_out):
            with rasterio.open(path,'w',driver='GTiff',height=lines,width=samples,count=1,dtype=dtype,
                               crs=crs,transform=transform,nodata=nodata_out,compress='deflate') as dst:
                dst.write(arr.astype(dtype),1)
        write_tif(GEO_DIR/'spectral_units_raw.tif',cluster_map,'int16',-1)
        write_tif(GEO_DIR/'spectral_units_consensus.tif',cluster_map_smooth,'int16',-1)
        write_tif(GEO_DIR/'anomaly_score.tif',np.where(np.isfinite(anomaly_map),anomaly_map,-9999),'float32',-9999)
        binary99=np.where(valid_mask,(anomaly_map>=np.nanpercentile(valid_scores,99)).astype(np.uint8),255)
        write_tif(GEO_DIR/'anomaly_binary_P99.tif',binary99,'uint8',255)
        geotiff_status.append('GeoTIFF export: SUCCESS')
    else:
        geotiff_status.append('GeoTIFF export skipped: map georeferencing fields unavailable.')
except Exception as e:
    geotiff_status.append(f'GeoTIFF export skipped/failed: {type(e).__name__}: {e}')

# ------------------------------------------------------------
# 9.10 Excel workbook containing all result tables
# ------------------------------------------------------------
excel_path=OUT_DIR/'ENMAP_PAPER_RESULTS_TABLES.xlsx'
tables_for_excel={
    'Scene_Summary':pd.DataFrame([summary]),
    'Wavelength_QC':pd.DataFrame({'band_1based':good_bands+1,'wavelength_nm':wl,'nodata_fraction':nodata_frac[good_bands],
                                  'gain':gain_good,'offset':offset_good,'model_weight':band_weight}),
    'K_Selection':kdf_all,
    'Ablation_Baselines':ablation_df,
    'Method_Ranking':rank_df,
    'Spatial_Coherence':pd.DataFrame([{'method':'AAMSAE raw','spatial_neighbor_agreement':coh_raw},
                                      {'method':'AAMSAE 3x3 consensus','spatial_neighbor_agreement':coh_smooth}]),
    'Cluster_Statistics':cluster_stats_df,
    'Anomaly_Thresholds':threshold_df,
    'Seed_Robustness':seed_robust_df,
    'Seed_Summary':seed_summary_df,
    'Bootstrap_CI':bootstrap_ci_df,
    'Absorption_Features':spectral_features_df,
    'Top_Anomalies':top_df,
    'Cluster_Mean_Spectra':pd.DataFrame(cluster_spec),
}
try:
    with pd.ExcelWriter(excel_path,engine='openpyxl') as writer:
        for sheet,df in tables_for_excel.items():
            df.to_excel(writer,sheet_name=sheet[:31],index=False)
except Exception as e:
    print('Excel export warning:',e)

# ------------------------------------------------------------
# 9.11 Paper notes, manifest, and final bundle
# ------------------------------------------------------------
notes=f"""
ENMAP AAMSAE PRO RESULTS NOTES
==============================
Scene: {HDR.parent.name}
Shape: {lines} rows × {samples} cols × {bands} original bands
Usable bands after automatic QC: {len(good_bands)}
Wavelength range used: {wl.min():.2f}–{wl.max():.2f} nm
Valid footprint pixels: {len(valid_coords)}
AAMSAE selected K: {FULL_K}

KEY INTERNAL METRICS
--------------------
AAMSAE silhouette: {aamsae_row.silhouette:.6f}
AAMSAE Davies-Bouldin: {aamsae_row.davies_bouldin:.6f}
AAMSAE Calinski-Harabasz: {aamsae_row.calinski_harabasz:.3f}
AAMSAE clustering stability ARI: {aamsae_row.cluster_stability_ARI:.6f}
Spatial neighbor agreement (raw): {coh_raw:.4f}
Spatial neighbor agreement (3x3 consensus): {coh_smooth:.4f}

AUTOMATIC CLUSTER-QUALITY WINNER
--------------------------------
{best_cluster_method}

SCIENTIFIC LIMITATION
---------------------
This scene has no independent ground-truth lithology/mineral labels.
Spectral Unit 0/1/... must NOT be renamed as a mineral or lithology without external validation.
Anomaly score is a prioritization score, not proof of mineralization.
Do not report OA/F1/accuracy unless independent labels are obtained.

RECOMMENDED EXTERNAL VALIDATION
-------------------------------
Use at least one:
1) geological/lithological polygon map,
2) known mineral occurrence coordinates,
3) field/XRD/ASD measurements,
4) independent mineral spectral library for post-hoc spectral identification.

PRO PACKAGE CONTENT
-------------------
- publication PNG + vector PDF figures,
- baseline/ablation/K-selection tables,
- cluster spatial statistics,
- anomaly threshold sensitivity,
- 10-seed robustness analysis,
- bootstrap 95% confidence intervals,
- candidate absorption-feature table,
- top anomaly coordinates and spectra,
- georeferenced GeoTIFF maps when rasterio/georeferencing is available,
- multi-sheet Excel workbook.
"""
(OUT_DIR/'README_PAPER_NOTES_PRO.txt').write_text(notes)
(OUT_DIR/'GEOTIFF_STATUS.txt').write_text('\n'.join(geotiff_status))

config={
    'seed':SEED,'train_samples':len(train_coords),'eval_samples':len(eval_coords),'full_epochs':FULL_EPOCHS,
    'ablation_epochs':ABLATION_EPOCHS,'batch_size':BATCH_SIZE,'latent_dim':LATENT_DIM,'mask_ratio':MASK_RATIO,
    'k_range':K_RANGE,'run_ablation':RUN_ABLATION,'map_stride':MAP_STRIDE,'device':str(DEVICE),
    'professional_outputs':True,'robustness_seeds':robust_seeds,'bootstrap_replicates':n_boot
}
(OUT_DIR/'run_config_PRO.json').write_text(json.dumps(config,indent=2))

manifest=[]
for p in sorted(OUT_DIR.rglob('*')):
    if p.is_file():
        manifest.append({'relative_path':str(p.relative_to(OUT_DIR)),'size_bytes':p.stat().st_size})
pd.DataFrame(manifest).to_csv(OUT_DIR/'RESULTS_MANIFEST.csv',index=False)



In [ ]:
# ============================================================
# 10) MULTI-BENCHMARK VALIDATION FOR HYPERSPECTRAL ANOMALY DETECTION
#     ABU (13 scenes) + HYDICE + San Diego (when downloadable)
# ============================================================
# IMPORTANT SCIENTIFIC PROTOCOL
# - Ground-truth labels are NEVER used to train the unsupervised models.
# - Ground truth is used only AFTER score generation for evaluation.
# - Fusion weights are fixed globally and are not tuned per scene.
# - "oracle_f1" is explicitly post-hoc and must not be described as an
#   unsupervised operating-point metric in the paper.

from scipy.io import loadmat
from scipy.ndimage import uniform_filter
from scipy.stats import rankdata, wilcoxon
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    precision_score, recall_score, f1_score, confusion_matrix
)
from sklearn.decomposition import PCA as SKPCA
from sklearn.covariance import LedoitWolf as SKLedoitWolf
import urllib.request

BENCH_DIR = WORK_ROOT / 'benchmark_data'
BENCH_OUT = OUT_DIR / 'benchmark_validation'
BENCH_FIG = BENCH_OUT / 'paper_ready_figures'
BENCH_CURVES = BENCH_OUT / 'curve_data'
BENCH_MAPS = BENCH_OUT / 'per_scene_maps'
for _d in (BENCH_DIR, BENCH_OUT, BENCH_FIG, BENCH_CURVES, BENCH_MAPS):
    _d.mkdir(parents=True, exist_ok=True)

BENCH_EPOCHS = int(os.environ.get('ENMAP_BENCH_EPOCHS', '15'))
BENCH_BATCH = int(os.environ.get('ENMAP_BENCH_BATCH', '512'))
BENCH_MASK_RATIO = float(os.environ.get('ENMAP_BENCH_MASK_RATIO', '0.35'))
BENCH_LATENT = int(os.environ.get('ENMAP_BENCH_LATENT', '24'))
BENCH_SEEDS = [int(x) for x in os.environ.get('ENMAP_BENCH_SEEDS', '42,123,2026').split(',') if x.strip()]
RUN_SAN_DIEGO = os.environ.get('ENMAP_RUN_SAN_DIEGO', '1') == '1'
RUN_HYDICE = os.environ.get('ENMAP_RUN_HYDICE', '1') == '1'

print('\n' + '='*80)
print('STARTING MULTI-BENCHMARK VALIDATION')
print('Benchmark epochs:', BENCH_EPOCHS, '| seeds:', BENCH_SEEDS)
print('='*80)

# ---------- Data acquisition ----------
ABU_SCENES = [
    *(f'abu-airport-{i}.mat' for i in range(1,5)),
    *(f'abu-beach-{i}.mat' for i in range(1,5)),
    *(f'abu-urban-{i}.mat' for i in range(1,6)),
]

ABU_GH_RAW = 'https://raw.githubusercontent.com/sxt1996/Airport-Beach-Urban-ABU/main/{name}'
ABU_WEIBLY = 'https://xudongkang.weebly.com/uploads/1/6/4/6/16465750/{name}'
HYDICE_URLS = [
    'https://raw.githubusercontent.com/sxt1996/HYDICE/main/HYDICE-urban.mat',
    'https://github.com/sxt1996/HYDICE/raw/main/HYDICE-urban.mat',
]
SANDIEGO_URLS = [
    'https://raw.githubusercontent.com/huyanning/Hyperspectral-Anomaly-Detection/master/Sandiego.mat',
    'https://github.com/huyanning/Hyperspectral-Anomaly-Detection/raw/master/Sandiego.mat',
]
PLANE_GT_URLS = [
    'https://raw.githubusercontent.com/huyanning/Hyperspectral-Anomaly-Detection/master/PlaneGT.mat',
    'https://github.com/huyanning/Hyperspectral-Anomaly-Detection/raw/master/PlaneGT.mat',
]


def _download_one(url, dest, timeout=90):
    """Small dependency-free downloader. Kaggle Internet must be ON."""
    req = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=timeout) as r, open(dest, 'wb') as f:
        shutil.copyfileobj(r, f, length=1024*1024)
    return dest.exists() and dest.stat().st_size > 1024


def download_with_fallback(urls, dest):
    if dest.exists() and dest.stat().st_size > 1024:
        return True
    last = None
    for url in urls:
        try:
            print('Downloading:', dest.name)
            if _download_one(url, dest):
                return True
        except Exception as e:
            last = e
            if dest.exists():
                try: dest.unlink()
                except Exception: pass
    print('WARNING: could not download', dest.name, '|', repr(last))
    return False


def acquire_benchmarks():
    files = []
    for name in ABU_SCENES:
        p = BENCH_DIR / name
        ok = download_with_fallback(
            [ABU_GH_RAW.format(name=name), ABU_WEIBLY.format(name=name)], p
        )
        if ok: files.append(('ABU', p, None))
    if RUN_HYDICE:
        p = BENCH_DIR / 'HYDICE-urban.mat'
        if download_with_fallback(HYDICE_URLS, p):
            files.append(('HYDICE', p, None))
    if RUN_SAN_DIEGO:
        sd = BENCH_DIR / 'Sandiego.mat'
        gt = BENCH_DIR / 'PlaneGT.mat'
        ok1 = download_with_fallback(SANDIEGO_URLS, sd)
        ok2 = download_with_fallback(PLANE_GT_URLS, gt)
        if ok1 and ok2:
            files.append(('SanDiego', sd, gt))
    return files

# ---------- MAT parsing ----------
def _numeric_arrays(mat):
    return {k:np.asarray(v) for k,v in mat.items()
            if not k.startswith('__') and isinstance(v, np.ndarray) and np.issubdtype(np.asarray(v).dtype, np.number)}


def load_benchmark(kind, path, gt_path=None):
    m = loadmat(path)
    arrs = _numeric_arrays(m)
    if kind in ('ABU','HYDICE'):
        cube = arrs.get('data', None)
        gt = arrs.get('map', None)
        if cube is None or gt is None:
            # robust fallback by dimensionality
            cube_candidates = [a for a in arrs.values() if a.ndim==3]
            map_candidates = [a for a in arrs.values() if a.ndim==2]
            if not cube_candidates or not map_candidates:
                raise ValueError(f'Could not identify data/map in {path.name}; keys={list(arrs)}')
            cube = max(cube_candidates, key=lambda a:a.size)
            gt = min(map_candidates, key=lambda a:a.size)
    elif kind == 'SanDiego':
        cube = arrs.get('Sandiego', None)
        if cube is None:
            cubes = [a for a in arrs.values() if a.ndim==3]
            if not cubes: raise ValueError('Sandiego cube not found')
            cube = max(cubes, key=lambda a:a.size)
        # Follow the source repository's published example protocol: 100x100 crop
        cube = np.asarray(cube, dtype=np.float32)[:100,:100,:]
        remove_bands = np.hstack((
            np.arange(0,6), np.arange(32,35), np.arange(93,97), np.arange(106,113),
            np.arange(152,166), np.arange(220,224)
        ))
        remove_bands = remove_bands[remove_bands < cube.shape[2]]
        cube = np.delete(cube, remove_bands, axis=2)
        gm = loadmat(gt_path)
        garr = _numeric_arrays(gm)
        gt = garr.get('PlaneGT', None)
        if gt is None:
            maps=[a for a in garr.values() if a.ndim==2]
            if not maps: raise ValueError('PlaneGT not found')
            gt=maps[0]
        gt=np.asarray(gt)[:100,:100]
    else:
        raise ValueError(kind)

    cube=np.asarray(cube, dtype=np.float32)
    gt=np.asarray(gt).squeeze()
    # Ensure H,W,B orientation by matching gt spatial dimensions.
    if cube.ndim != 3 or gt.ndim != 2:
        raise ValueError(f'Unexpected shapes cube={cube.shape}, gt={gt.shape}')
    if cube.shape[:2] != gt.shape:
        # Try common MATLAB B,H,W arrangement.
        if cube.shape[1:] == gt.shape:
            cube=np.moveaxis(cube,0,-1)
        elif (cube.shape[0],cube.shape[2]) == gt.shape:
            cube=np.moveaxis(cube,1,-1)
        else:
            raise ValueError(f'Spatial mismatch cube={cube.shape}, gt={gt.shape}')

    # Binary ground truth; any positive label counts as anomaly.
    gt=(gt>0).astype(np.uint8)
    finite=np.all(np.isfinite(cube),axis=2)
    # Replace rare invalid entries bandwise so methods receive the same pixels.
    for b in range(cube.shape[2]):
        band=cube[:,:,b]
        bad=~np.isfinite(band)
        if bad.any():
            med=np.nanmedian(band)
            band[bad]=med if np.isfinite(med) else 0.0
            cube[:,:,b]=band
    return cube, gt

# ---------- Robust preprocessing ----------
def robust_band_scale(cube):
    H,W,B=cube.shape
    X=cube.reshape(-1,B).astype(np.float32)
    med=np.median(X,axis=0)
    q25=np.percentile(X,25,axis=0); q75=np.percentile(X,75,axis=0)
    scale=(q75-q25)/1.349
    std=X.std(axis=0)
    scale=np.where(scale>1e-8, scale, np.where(std>1e-8,std,1.0))
    X=(X-med)/scale
    X=np.clip(X,-8,8).astype(np.float32)
    return X, med.astype(np.float32), scale.astype(np.float32)


def ecdf_rank_score(x):
    x=np.asarray(x,dtype=float).ravel()
    if len(x)<=1: return np.zeros_like(x)
    return (rankdata(x,method='average')-1)/(len(x)-1)


def classical_rx(X):
    # Shrinkage covariance avoids singularity in high-dimensional HSI.
    cov=SKLedoitWolf().fit(X)
    d=X-cov.location_
    prec=cov.precision_
    return np.einsum('ij,jk,ik->i', d, prec, d)


def pca_rx(X, ncomp=20, seed=42):
    n=min(ncomp, X.shape[1], max(2,X.shape[0]-1))
    P=SKPCA(n_components=n,random_state=seed).fit_transform(X)
    return classical_rx(P)

# ---------- Generic self-supervised spectral model ----------
class BenchSpectralMAE(nn.Module):
    def __init__(self, n_bands, latent_dim=24):
        super().__init__()
        self.n_bands=n_bands
        self.encoder=nn.Sequential(
            nn.Conv1d(3,32,7,padding=3), nn.BatchNorm1d(32), nn.GELU(),
            nn.Conv1d(32,64,5,stride=2,padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64,96,5,stride=2,padding=2), nn.BatchNorm1d(96), nn.GELU(),
            nn.Conv1d(96,128,3,stride=2,padding=1), nn.BatchNorm1d(128), nn.GELU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.to_latent=nn.Linear(128,latent_dim)
        self.decoder=nn.Sequential(
            nn.Linear(latent_dim,128),nn.GELU(),
            nn.Linear(128,256),nn.GELU(),
            nn.Linear(256,n_bands)
        )
        pos=np.linspace(0,1,n_bands,dtype=np.float32)
        self.register_buffer('pos',torch.tensor(pos).view(1,1,-1))
    def make_input(self,x,mask=None):
        if mask is None: mask=torch.zeros_like(x,dtype=torch.bool)
        xm=x.masked_fill(mask,0.0)
        p=self.pos.expand(x.shape[0],-1,-1)
        return torch.cat([xm[:,None,:],mask.float()[:,None,:],p],1)
    def encode(self,x):
        h=self.encoder(self.make_input(x,None)).squeeze(-1)
        return self.to_latent(h)
    def forward(self,x,mask=None):
        h=self.encoder(self.make_input(x,mask)).squeeze(-1)
        z=self.to_latent(h)
        return self.decoder(z),z


def train_bench_model(X, mode='masked', seed=42, epochs=15):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    model=BenchSpectralMAE(X.shape[1],BENCH_LATENT).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=2e-3,weight_decay=1e-4)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max(1,epochs),eta_min=2e-4)
    # Use all pixels, labels remain unknown to the model. Cap only for very large scenes.
    rng_local=np.random.default_rng(seed)
    if len(X)>30000:
        idx=rng_local.choice(len(X),30000,replace=False); Xt=X[idx]
    else:
        Xt=X
    dl=DataLoader(TensorDataset(torch.from_numpy(Xt)),batch_size=BENCH_BATCH,shuffle=True,num_workers=0)
    losses=[]; best=np.inf; state=None
    t0=time.perf_counter()
    for ep in range(epochs):
        model.train(); s=0.0; n=0
        for (xb,) in dl:
            xb=xb.to(DEVICE)
            if mode=='plain':
                mask=torch.zeros_like(xb,dtype=torch.bool)
            else:
                mask=torch.rand_like(xb)<BENCH_MASK_RATIO
                empty=~mask.any(1)
                if empty.any(): mask[empty,0]=True
            rec,_=model(xb,mask)
            sq=(rec-xb)**2
            if mode=='plain':
                loss=sq.mean()
            else:
                # Masked reconstruction + global bottleneck + contiguous-band derivative fidelity.
                recon=(sq*mask.float()).sum()/mask.float().sum().clamp_min(1.0)
                global_loss=sq.mean()
                if mode=='derivative':
                    dm=(mask[:,1:]|mask[:,:-1]).float()
                    dr=(rec[:,1:]-rec[:,:-1])-(xb[:,1:]-xb[:,:-1])
                    deriv=((dr**2)*dm).sum()/dm.sum().clamp_min(1.0)
                    loss=recon+0.12*deriv+0.03*global_loss
                else:
                    loss=recon+0.03*global_loss
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
            opt.step(); s+=float(loss.item())*len(xb); n+=len(xb)
        sch.step(); el=s/max(n,1); losses.append(el)
        if el<best:
            best=el; state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    if state is not None: model.load_state_dict(state)
    model.to(DEVICE).eval()
    return model, losses, time.perf_counter()-t0


@torch.no_grad()
def bench_encode_reconstruct(model,X,batch=4096):
    Z=[]; R=[]
    t0=time.perf_counter()
    for s in range(0,len(X),batch):
        xb=torch.from_numpy(X[s:s+batch]).to(DEVICE)
        rec,z=model(xb,None)
        Z.append(z.cpu().numpy()); R.append(rec.cpu().numpy())
    return np.vstack(Z),np.vstack(R),time.perf_counter()-t0


def latent_rx(Z):
    # Standardize dimensions before shrinkage RX.
    mu=Z.mean(0); sd=Z.std(0); sd=np.where(sd>1e-8,sd,1.0)
    return classical_rx(((Z-mu)/sd).astype(np.float32))


def local_latent_contrast(Z,H,W,size=5):
    z=Z.reshape(H,W,-1).astype(np.float32)
    loc=np.empty_like(z)
    for j in range(z.shape[2]):
        loc[:,:,j]=uniform_filter(z[:,:,j],size=size,mode='reflect')
    d=z-loc
    return np.sqrt(np.mean(d*d,axis=2)).ravel()


def spectral_recon_score(X,R):
    return np.mean((R-X)**2,axis=1)


def fixed_fusion(rec, lrx, local):
    # Fixed, scene-independent rank fusion. No labels are used here.
    return 0.30*ecdf_rank_score(rec)+0.45*ecdf_rank_score(lrx)+0.25*ecdf_rank_score(local)


def metrics_from_score(y,score):
    y=np.asarray(y).astype(np.uint8).ravel(); s=np.asarray(score,float).ravel()
    if len(np.unique(y))<2:
        return None
    auc=float(roc_auc_score(y,s))
    ap=float(average_precision_score(y,s))
    # Fixed unsupervised threshold at P99.
    thr99=float(np.quantile(s,0.99)); p99=(s>=thr99).astype(np.uint8)
    f1p=float(f1_score(y,p99,zero_division=0)); pp=float(precision_score(y,p99,zero_division=0)); rp=float(recall_score(y,p99,zero_division=0))
    # Post-hoc oracle F1 for diagnostic comparison only.
    prec,rec,thr=precision_recall_curve(y,s)
    f=2*prec*rec/(prec+rec+1e-12)
    j=int(np.nanargmax(f))
    oracle=float(f[j])
    oracle_thr=float(thr[min(j,len(thr)-1)]) if len(thr) else float(np.median(s))
    # FPR at approximately 95% TPR.
    fpr,tpr,_=roc_curve(y,s)
    ids=np.where(tpr>=0.95)[0]
    fpr95=float(fpr[ids[0]]) if len(ids) else np.nan
    return dict(roc_auc=auc,pr_auc=ap,f1_p99=f1p,precision_p99=pp,recall_p99=rp,
                oracle_f1=oracle,oracle_threshold=oracle_thr,fpr_at_95_tpr=fpr95,
                anomaly_prevalence=float(y.mean()))


def save_curve_csv(dataset, method, y, score):
    fpr,tpr,rthr=roc_curve(y,score)
    p,r,pthr=precision_recall_curve(y,score)
    pd.DataFrame({'fpr':fpr,'tpr':tpr,'threshold':rthr}).to_csv(BENCH_CURVES/f'{dataset}__{method}__roc.csv',index=False)
    # PR has one more precision/recall point than thresholds.
    pthr2=np.r_[pthr,np.nan]
    pd.DataFrame({'recall':r,'precision':p,'threshold':pthr2}).to_csv(BENCH_CURVES/f'{dataset}__{method}__pr.csv',index=False)


def pseudo_rgb(cube):
    B=cube.shape[2]
    ids=[max(0,min(B-1,int(B*q))) for q in (0.75,0.5,0.25)]
    rgb=cube[:,:,ids].astype(np.float32)
    out=np.zeros_like(rgb)
    for c in range(3):
        a=rgb[:,:,c]; lo,hi=np.percentile(a,[2,98])
        out[:,:,c]=np.clip((a-lo)/(hi-lo+1e-8),0,1)
    return out


def save_bench_scene_figure(name,cube,gt,scores):
    methods=['RX','PCA-RX','MSAE-LatentRX','Proposed-SSMSAE']
    fig,axs=plt.subplots(2,3,figsize=(14,8))
    axs=axs.ravel()
    axs[0].imshow(pseudo_rgb(cube)); axs[0].set_title('Pseudo RGB')
    axs[1].imshow(gt,cmap='gray'); axs[1].set_title('Ground Truth')
    for ax,m in zip(axs[2:],methods):
        im=ax.imshow(scores[m].reshape(gt.shape),cmap='magma')
        ax.set_title(m); fig.colorbar(im,ax=ax,fraction=0.046,pad=0.04)
    for ax in axs: ax.axis('off')
    fig.suptitle(name)
    fig.tight_layout(rect=[0,0,1,0.96])
    fig.savefig(BENCH_MAPS/f'{name}_detection_panel.png',dpi=300,bbox_inches='tight')
    plt.close(fig)

# ---------- Run benchmark scenes ----------
bench_files=acquire_benchmarks()
if len(bench_files) < 3:
    msg=(
        'Benchmark download did not provide enough scenes. Enable Internet in Kaggle Settings and rerun.\n'
        'The EnMAP pipeline is still valid, but multi-benchmark quantitative validation was not completed.'
    )
    (BENCH_OUT/'BENCHMARK_DOWNLOAD_FAILED.txt').write_text(msg)
    print('WARNING:',msg)
    benchmark_metrics=pd.DataFrame()
    seed_metrics=pd.DataFrame()
else:
    all_rows=[]; seed_rows=[]; ablation_rows=[]
    dataset_info=[]
    selected_curves={}
    for di,(kind,path,gt_path) in enumerate(bench_files,1):
        name=path.stem
        print(f'\n[{di}/{len(bench_files)}] Benchmark:',name)
        try:
            cube,gt=load_benchmark(kind,path,gt_path)
        except Exception as e:
            print('SKIP due to loader error:',repr(e)); continue
        H,W,B=cube.shape; y=gt.ravel().astype(np.uint8)
        if y.sum()==0 or y.sum()==len(y):
            print('SKIP: ground truth is not binary/informative'); continue
        X,_,_=robust_band_scale(cube)
        dataset_info.append({'dataset':name,'family':kind,'height':H,'width':W,'bands':B,
                             'pixels':H*W,'anomaly_pixels':int(y.sum()),'anomaly_prevalence':float(y.mean())})

        scene_scores={}; runtimes={}; train_times={}; params={}
        # Classical RX
        t0=time.perf_counter(); scene_scores['RX']=classical_rx(X); runtimes['RX']=time.perf_counter()-t0
        # PCA-RX
        t0=time.perf_counter(); scene_scores['PCA-RX']=pca_rx(X,20,SEED); runtimes['PCA-RX']=time.perf_counter()-t0

        # Plain AE (seed 42)
        ae,ae_loss,ae_train=train_bench_model(X,'plain',SEED,BENCH_EPOCHS)
        Zae,Rae,ae_inf=bench_encode_reconstruct(ae,X)
        scene_scores['AE-Reconstruction']=spectral_recon_score(X,Rae)
        train_times['AE-Reconstruction']=ae_train; runtimes['AE-Reconstruction']=ae_inf
        params['AE-Reconstruction']=sum(p.numel() for p in ae.parameters())
        del ae,Zae,Rae
        if torch.cuda.is_available(): torch.cuda.empty_cache()

        # Standard masked AE
        msae,ms_loss,ms_train=train_bench_model(X,'masked',SEED,BENCH_EPOCHS)
        Zm,Rm,ms_inf=bench_encode_reconstruct(msae,X)
        rec_m=spectral_recon_score(X,Rm); lrx_m=latent_rx(Zm); loc_m=local_latent_contrast(Zm,H,W,5)
        scene_scores['MSAE-Reconstruction']=rec_m
        scene_scores['MSAE-LatentRX']=lrx_m
        scene_scores['MSAE-Fusion']=fixed_fusion(rec_m,lrx_m,loc_m)
        for m in ['MSAE-Reconstruction','MSAE-LatentRX','MSAE-Fusion']:
            train_times[m]=ms_train; runtimes[m]=ms_inf; params[m]=sum(p.numel() for p in msae.parameters())

        # Derivative-aware MSAE = proposed encoder; repeat across seeds for robustness.
        proposed_seed_scores=[]
        for si,s in enumerate(BENCH_SEEDS):
            dm,dm_loss,dm_train=train_bench_model(X,'derivative',s,BENCH_EPOCHS)
            Zd,Rd,dm_inf=bench_encode_reconstruct(dm,X)
            rec_d=spectral_recon_score(X,Rd); lrx_d=latent_rx(Zd); loc_d=local_latent_contrast(Zd,H,W,5)
            fused=fixed_fusion(rec_d,lrx_d,loc_d)
            met=metrics_from_score(y,fused)
            if met:
                seed_rows.append({'dataset':name,'seed':s,**met})
            proposed_seed_scores.append(fused)
            if si==0:
                scene_scores['DMSAE-Reconstruction']=rec_d
                scene_scores['DMSAE-LatentRX']=lrx_d
                scene_scores['Proposed-SSMSAE']=fused
                train_times['DMSAE-Reconstruction']=dm_train; runtimes['DMSAE-Reconstruction']=dm_inf
                train_times['DMSAE-LatentRX']=dm_train; runtimes['DMSAE-LatentRX']=dm_inf
                train_times['Proposed-SSMSAE']=dm_train; runtimes['Proposed-SSMSAE']=dm_inf
                pcount=sum(p.numel() for p in dm.parameters())
                params['DMSAE-Reconstruction']=pcount; params['DMSAE-LatentRX']=pcount; params['Proposed-SSMSAE']=pcount
                # Component ablation uses the SAME learned encoder and fixed cues.
                for an,sc in {
                    'DMSAE Rec only':rec_d,
                    'DMSAE Latent-RX only':lrx_d,
                    'DMSAE Local only':loc_d,
                    'Rec + LatentRX':0.40*ecdf_rank_score(rec_d)+0.60*ecdf_rank_score(lrx_d),
                    'Rec + Local':0.55*ecdf_rank_score(rec_d)+0.45*ecdf_rank_score(loc_d),
                    'LatentRX + Local':0.65*ecdf_rank_score(lrx_d)+0.35*ecdf_rank_score(loc_d),
                    'Full Proposed':fused,
                }.items():
                    mm=metrics_from_score(y,sc)
                    if mm: ablation_rows.append({'dataset':name,'variant':an,**mm})
            del dm,Zd,Rd
            if torch.cuda.is_available(): torch.cuda.empty_cache()

        # Evaluate every method; ground truth enters ONLY here.
        for method,score in scene_scores.items():
            met=metrics_from_score(y,score)
            if met is None: continue
            all_rows.append({'dataset':name,'family':kind,'method':method,
                             'score_train_seconds':float(train_times.get(method,0.0)),
                             'score_inference_seconds':float(runtimes.get(method,0.0)),
                             'parameter_count':int(params.get(method,0)),**met})
            save_curve_csv(name,method,y,score)
        # Save the representative scene panel for every scene.
        required=['RX','PCA-RX','MSAE-LatentRX','Proposed-SSMSAE']
        if all(m in scene_scores for m in required):
            save_bench_scene_figure(name,cube,gt,scene_scores)

        del msae,Zm,Rm
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    benchmark_metrics=pd.DataFrame(all_rows)
    seed_metrics=pd.DataFrame(seed_rows)
    ablation_df=pd.DataFrame(ablation_rows)
    dataset_info_df=pd.DataFrame(dataset_info)
    benchmark_metrics.to_csv(BENCH_OUT/'10_benchmark_all_metrics.csv',index=False)
    seed_metrics.to_csv(BENCH_OUT/'11_proposed_seed_robustness.csv',index=False)
    ablation_df.to_csv(BENCH_OUT/'12_proposed_ablation.csv',index=False)
    dataset_info_df.to_csv(BENCH_OUT/'13_benchmark_dataset_summary.csv',index=False)

    if not benchmark_metrics.empty:
        # Summary + average ranks
        summary=benchmark_metrics.groupby('method').agg(
            scenes=('dataset','nunique'), roc_auc_mean=('roc_auc','mean'),roc_auc_std=('roc_auc','std'),
            pr_auc_mean=('pr_auc','mean'),pr_auc_std=('pr_auc','std'),
            f1_p99_mean=('f1_p99','mean'),oracle_f1_mean=('oracle_f1','mean'),
            fpr95_mean=('fpr_at_95_tpr','mean'),
            inference_seconds_mean=('score_inference_seconds','mean'),
            train_seconds_mean=('score_train_seconds','mean'),
            parameter_count=('parameter_count','max')
        ).reset_index()
        rank_piv=benchmark_metrics.pivot(index='dataset',columns='method',values='roc_auc')
        ranks=rank_piv.rank(axis=1,ascending=False,method='average')
        mean_rank=ranks.mean().rename('mean_roc_auc_rank').reset_index()
        summary=summary.merge(mean_rank,on='method',how='left').sort_values(['roc_auc_mean','pr_auc_mean'],ascending=False)
        summary.to_csv(BENCH_OUT/'14_benchmark_method_summary.csv',index=False)
        ranks.to_csv(BENCH_OUT/'15_per_scene_method_ranks.csv')

        # Wilcoxon paired tests: Proposed vs each baseline, with Holm adjustment.
        prop='Proposed-SSMSAE'; tests=[]
        if prop in rank_piv.columns:
            for metric in ['roc_auc','pr_auc']:
                pv=benchmark_metrics.pivot(index='dataset',columns='method',values=metric)
                if prop not in pv: continue
                for m in pv.columns:
                    if m==prop: continue
                    pair=pv[[prop,m]].dropna()
                    if len(pair)<5: continue
                    diff=pair[prop]-pair[m]
                    try:
                        stat,p=wilcoxon(diff,alternative='greater',zero_method='wilcox')
                    except Exception:
                        stat,p=np.nan,1.0
                    tests.append({'metric':metric,'comparison':f'{prop} > {m}','n':len(pair),
                                  'mean_difference':float(diff.mean()),'median_difference':float(diff.median()),
                                  'wilcoxon_stat':float(stat) if np.isfinite(stat) else np.nan,'p_raw':float(p)})
        tests_df=pd.DataFrame(tests)
        if not tests_df.empty:
            # Holm correction within each metric family.
            tests_df['p_holm']=np.nan
            for metric,g in tests_df.groupby('metric'):
                ids=g.index.to_list(); order=sorted(ids,key=lambda i:tests_df.loc[i,'p_raw'])
                running=0.0
                m=len(order)
                for r,i in enumerate(order):
                    adj=min(1.0,(m-r)*tests_df.loc[i,'p_raw'])
                    running=max(running,adj); tests_df.loc[i,'p_holm']=running
            tests_df['significant_0_05']=tests_df.p_holm<0.05
        tests_df.to_csv(BENCH_OUT/'16_wilcoxon_significance_tests.csv',index=False)

        # Bootstrap CI across scenes for mean AUC/AP (scene-level resampling).
        boot_rows=[]; brng=np.random.default_rng(2026)
        for method,g in benchmark_metrics.groupby('method'):
            for metric in ['roc_auc','pr_auc']:
                vals=g[metric].dropna().to_numpy(float)
                if len(vals)<3: continue
                boots=np.array([brng.choice(vals,len(vals),replace=True).mean() for _ in range(3000)])
                boot_rows.append({'method':method,'metric':metric,'mean':vals.mean(),
                                  'ci95_low':np.quantile(boots,0.025),'ci95_high':np.quantile(boots,0.975),'n_scenes':len(vals)})
        boot_df=pd.DataFrame(boot_rows)
        boot_df.to_csv(BENCH_OUT/'17_bootstrap_scene_level_ci.csv',index=False)

        # ---------- Publication figures ----------
        def _save_bench_fig(fig,stem):
            fig.savefig(BENCH_FIG/f'{stem}.png',dpi=350,bbox_inches='tight')
            fig.savefig(BENCH_FIG/f'{stem}.pdf',bbox_inches='tight')
            plt.close(fig)

        # Fig B01: AUC heatmap
        hp=benchmark_metrics.pivot(index='dataset',columns='method',values='roc_auc')
        ordered=summary.method.tolist(); hp=hp[[m for m in ordered if m in hp.columns]]
        fig,ax=plt.subplots(figsize=(max(10,1.35*len(hp.columns)),max(7,0.45*len(hp))))
        im=ax.imshow(hp.values,aspect='auto',vmin=0.5,vmax=1.0,cmap='viridis')
        ax.set_xticks(range(len(hp.columns)),hp.columns,rotation=45,ha='right')
        ax.set_yticks(range(len(hp.index)),hp.index)
        ax.set_title('ROC-AUC across hyperspectral anomaly benchmarks')
        fig.colorbar(im,ax=ax,label='ROC-AUC')
        _save_bench_fig(fig,'fig_b01_auc_heatmap')

        # Fig B02: Mean AUC with bootstrap CI
        auc_ci=boot_df[boot_df.metric=='roc_auc'].set_index('method').reindex(ordered).dropna()
        fig,ax=plt.subplots(figsize=(10,6))
        y=np.arange(len(auc_ci)); means=auc_ci['mean'].values
        err=np.vstack([means-auc_ci.ci95_low.values,auc_ci.ci95_high.values-means])
        ax.errorbar(means,y,xerr=err,fmt='o',capsize=4)
        ax.set_yticks(y,auc_ci.index); ax.invert_yaxis(); ax.set_xlim(0.45,1.01)
        ax.set_xlabel('Mean ROC-AUC (95% bootstrap CI)'); ax.set_title('Cross-scene detection performance')
        ax.grid(axis='x',alpha=0.25)
        _save_bench_fig(fig,'fig_b02_mean_auc_ci')

        # Fig B03: PR-AUC mean and rank
        prsum=summary.sort_values('pr_auc_mean',ascending=True)
        fig,ax=plt.subplots(figsize=(10,6))
        ax.barh(prsum.method,prsum.pr_auc_mean)
        ax.set_xlabel('Mean PR-AUC'); ax.set_title('Precision–recall performance across benchmark scenes')
        ax.grid(axis='x',alpha=0.25)
        _save_bench_fig(fig,'fig_b03_mean_pr_auc')

        # Fig B04: Average ROC-AUC ranks
        rr=summary.sort_values('mean_roc_auc_rank',ascending=False)
        fig,ax=plt.subplots(figsize=(10,6))
        ax.barh(rr.method,rr.mean_roc_auc_rank)
        ax.set_xlabel('Mean rank (1 = best)'); ax.set_title('Average method rank across scenes')
        _save_bench_fig(fig,'fig_b04_average_rank')

        # Fig B05: Ablation
        if not ablation_df.empty:
            aa=ablation_df.groupby('variant')[['roc_auc','pr_auc']].mean().sort_values('roc_auc')
            fig,ax=plt.subplots(figsize=(10,6))
            yy=np.arange(len(aa)); h=0.36
            ax.barh(yy-h/2,aa.roc_auc,height=h,label='ROC-AUC')
            ax.barh(yy+h/2,aa.pr_auc,height=h,label='PR-AUC')
            ax.set_yticks(yy,aa.index); ax.set_xlabel('Mean metric'); ax.set_title('Proposed detector component ablation')
            ax.legend(); ax.grid(axis='x',alpha=0.25)
            _save_bench_fig(fig,'fig_b05_ablation')

        # Fig B06: Runtime vs ROC-AUC
        fig,ax=plt.subplots(figsize=(9,6))
        for _,r in summary.iterrows():
            x=max(r.inference_seconds_mean,1e-5); yv=r.roc_auc_mean
            ax.scatter(x,yv,s=55); ax.annotate(r.method,(x,yv),xytext=(4,4),textcoords='offset points',fontsize=8)
        ax.set_xscale('log'); ax.set_xlabel('Mean inference time per scene (s, log scale)'); ax.set_ylabel('Mean ROC-AUC')
        ax.set_title('Accuracy–efficiency trade-off'); ax.grid(alpha=0.25)
        _save_bench_fig(fig,'fig_b06_runtime_vs_auc')

        # Fig B07: proposed seed robustness
        if not seed_metrics.empty:
            sr=seed_metrics.groupby('seed')[['roc_auc','pr_auc']].mean()
            fig,ax=plt.subplots(figsize=(8,5))
            x=np.arange(len(sr)); w=0.35
            ax.bar(x-w/2,sr.roc_auc,width=w,label='ROC-AUC'); ax.bar(x+w/2,sr.pr_auc,width=w,label='PR-AUC')
            ax.set_xticks(x,[str(i) for i in sr.index]); ax.set_xlabel('Random seed'); ax.set_ylabel('Cross-scene mean')
            ax.set_title('Proposed-method robustness to initialization'); ax.legend(); ax.grid(axis='y',alpha=0.25)
            _save_bench_fig(fig,'fig_b07_seed_robustness')

        # Fig B08: boxplot scene distributions
        methods=ordered
        vals=[benchmark_metrics.loc[benchmark_metrics.method==m,'roc_auc'].dropna().values for m in methods]
        fig,ax=plt.subplots(figsize=(max(10,1.1*len(methods)),6))
        ax.boxplot(vals,tick_labels=methods,showmeans=True)
        ax.tick_params(axis='x',rotation=45); ax.set_ylabel('ROC-AUC'); ax.set_title('Scene-level ROC-AUC distributions')
        ax.grid(axis='y',alpha=0.25)
        _save_bench_fig(fig,'fig_b08_auc_boxplots')

        # Fig B09/B10: representative ROC and PR curves for 3 scenes
        scene_order=benchmark_metrics.dataset.unique().tolist()
        reps=[]
        for token in ['airport','beach','urban','HYDICE','Sandiego']:
            hit=next((s for s in scene_order if token.lower() in s.lower() and s not in reps),None)
            if hit: reps.append(hit)
            if len(reps)==3: break
        methods_curves=['RX','PCA-RX','MSAE-LatentRX','Proposed-SSMSAE']
        if reps:
            fig,axs=plt.subplots(1,len(reps),figsize=(6*len(reps),5),squeeze=False)
            for ax,ds in zip(axs[0],reps):
                for m in methods_curves:
                    fp=BENCH_CURVES/f'{ds}__{m}__roc.csv'
                    if fp.exists():
                        c=pd.read_csv(fp); aucv=benchmark_metrics[(benchmark_metrics.dataset==ds)&(benchmark_metrics.method==m)].roc_auc
                        lab=f'{m} ({aucv.iloc[0]:.3f})' if len(aucv) else m
                        ax.plot(c.fpr,c.tpr,label=lab)
                ax.plot([0,1],[0,1],'--',linewidth=1); ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
                ax.set_title(ds); ax.legend(fontsize=8); ax.grid(alpha=0.2)
            fig.suptitle('Representative ROC curves'); fig.tight_layout(rect=[0,0,1,0.95]); _save_bench_fig(fig,'fig_b09_representative_roc')

            fig,axs=plt.subplots(1,len(reps),figsize=(6*len(reps),5),squeeze=False)
            for ax,ds in zip(axs[0],reps):
                for m in methods_curves:
                    fp=BENCH_CURVES/f'{ds}__{m}__pr.csv'
                    if fp.exists():
                        c=pd.read_csv(fp); apv=benchmark_metrics[(benchmark_metrics.dataset==ds)&(benchmark_metrics.method==m)].pr_auc
                        lab=f'{m} ({apv.iloc[0]:.3f})' if len(apv) else m
                        ax.plot(c.recall,c.precision,label=lab)
                ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.set_title(ds); ax.legend(fontsize=8); ax.grid(alpha=0.2)
            fig.suptitle('Representative precision–recall curves'); fig.tight_layout(rect=[0,0,1,0.95]); _save_bench_fig(fig,'fig_b10_representative_pr')

        # Fig B11: one-page benchmark summary
        fig=plt.figure(figsize=(14,9))
        gs=fig.add_gridspec(2,2,hspace=0.35,wspace=0.35)
        ax=fig.add_subplot(gs[0,0]); top=summary.sort_values('roc_auc_mean').tail(6)
        ax.barh(top.method,top.roc_auc_mean); ax.set_xlim(0.5,1); ax.set_title('Mean ROC-AUC'); ax.grid(axis='x',alpha=.2)
        ax=fig.add_subplot(gs[0,1]); top=summary.sort_values('pr_auc_mean').tail(6)
        ax.barh(top.method,top.pr_auc_mean); ax.set_title('Mean PR-AUC'); ax.grid(axis='x',alpha=.2)
        ax=fig.add_subplot(gs[1,0]); rr=summary.sort_values('mean_roc_auc_rank').head(6)
        ax.barh(rr.method[::-1],rr.mean_roc_auc_rank[::-1]); ax.set_title('Mean rank (lower is better)')
        ax=fig.add_subplot(gs[1,1]);
        if not seed_metrics.empty:
            tmp=seed_metrics.groupby('seed').roc_auc.mean(); ax.plot(tmp.index.astype(str),tmp.values,marker='o'); ax.set_ylim(0.5,1.0)
            ax.set_title('Proposed seed robustness'); ax.set_xlabel('Seed'); ax.set_ylabel('Mean ROC-AUC'); ax.grid(alpha=.2)
        else:
            ax.axis('off'); ax.text(.5,.5,'Seed robustness unavailable',ha='center',va='center')
        fig.suptitle('Multi-benchmark hyperspectral anomaly-detection validation',fontsize=16)
        _save_bench_fig(fig,'fig_b11_benchmark_summary_panel')

        # Multi-sheet workbook for the paper.
        bench_excel=BENCH_OUT/'BENCHMARK_PAPER_RESULTS.xlsx'
        with pd.ExcelWriter(bench_excel,engine='openpyxl') as w:
            dataset_info_df.to_excel(w,sheet_name='Dataset Summary',index=False)
            benchmark_metrics.to_excel(w,sheet_name='All Metrics',index=False)
            summary.to_excel(w,sheet_name='Method Summary',index=False)
            ranks.reset_index().to_excel(w,sheet_name='Per Scene Ranks',index=False)
            seed_metrics.to_excel(w,sheet_name='Seed Robustness',index=False)
            ablation_df.to_excel(w,sheet_name='Ablation',index=False)
            tests_df.to_excel(w,sheet_name='Wilcoxon Tests',index=False)
            boot_df.to_excel(w,sheet_name='Bootstrap CI',index=False)

        # Automated scientific decision: does the proposed method actually win?
        best_auc=summary.sort_values('roc_auc_mean',ascending=False).iloc[0]
        best_pr=summary.sort_values('pr_auc_mean',ascending=False).iloc[0]
        prop_row=summary[summary.method=='Proposed-SSMSAE']
        lines=[]
        lines.append('V3 MULTI-BENCHMARK SCIENTIFIC DECISION')
        lines.append('='*48)
        lines.append(f'Number of evaluated benchmark scenes: {benchmark_metrics.dataset.nunique()}')
        lines.append(f'Best mean ROC-AUC method: {best_auc.method} ({best_auc.roc_auc_mean:.4f})')
        lines.append(f'Best mean PR-AUC method: {best_pr.method} ({best_pr.pr_auc_mean:.4f})')
        if len(prop_row):
            r=prop_row.iloc[0]
            lines.append(f'Proposed-SSMSAE mean ROC-AUC: {r.roc_auc_mean:.4f} ± {r.roc_auc_std:.4f}')
            lines.append(f'Proposed-SSMSAE mean PR-AUC: {r.pr_auc_mean:.4f} ± {r.pr_auc_std:.4f}')
            lines.append(f'Proposed-SSMSAE mean ROC-AUC rank: {r.mean_roc_auc_rank:.3f}')
            if best_auc.method=='Proposed-SSMSAE' and best_pr.method=='Proposed-SSMSAE':
                lines.append('DECISION: The proposed detector ranks first on both primary cross-scene metrics.')
            else:
                lines.append('DECISION: Do NOT claim overall superiority. Reframe the paper or improve the model before submission.')
        lines.append('Ground truth was used only for post-hoc evaluation, not model training or fusion-weight tuning.')
        lines.append('oracle_f1 is post-hoc and must be labeled as such in the manuscript.')
        (BENCH_OUT/'SCIENTIFIC_DECISION_V3.txt').write_text('\n'.join(lines))
        print('\n'.join(lines))



In [ ]:
# ============================================================
# 11) FINAL V3 RESULTS BUNDLE
# ============================================================
# Add a compact manifest describing exactly what can be cited in the paper.
manifest = {
    'enmap_results_directory': str(OUT_DIR),
    'benchmark_results_directory': str(BENCH_OUT),
    'benchmark_primary_metrics': ['ROC-AUC','PR-AUC'],
    'benchmark_secondary_metrics': ['F1@P99','Precision@P99','Recall@P99','FPR@95TPR','oracle F1 (post-hoc only)'],
    'benchmark_protocol': 'Unsupervised training; ground truth used only after score generation for evaluation.',
    'proposed_detector': 'Derivative-aware masked spectral autoencoder + fixed rank fusion of reconstruction, latent RX, and local latent contrast.',
    'fusion_weights': {'reconstruction':0.30,'latent_rx':0.45,'local_latent_contrast':0.25},
    'random_seeds': BENCH_SEEDS,
    'warning': 'Do not assign mineral/lithology names to EnMAP spectral units or anomalies without independent geological validation.'
}
(OUT_DIR/'V3_EXPERIMENT_MANIFEST.json').write_text(json.dumps(manifest,indent=2))

final_bundle=WORK_ROOT/'ENMAP_SSMAE_V3_FINAL_RESULTS_BUNDLE.zip'
if final_bundle.exists(): final_bundle.unlink()
shutil.make_archive(str(final_bundle.with_suffix('')), 'zip', root_dir=OUT_DIR)

print('\n'+'='*80)
print('V3 FINAL PIPELINE DONE')
print('EnMAP figures/results:', OUT_DIR)
print('Benchmark validation:', BENCH_OUT)
print('SEND ME THIS FILE:')
print(final_bundle)
print('='*80)
